<a href="https://colab.research.google.com/github/khalidalzzahranii-eng/customer-support-intelligence-platform-sdaia-/blob/main/Customer_Support_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# Setup & Imports
# =========================================================

from pathlib import Path
from datetime import datetime, timedelta, timezone
import importlib.util
import json
import random
import re
import subprocess
import sys
import uuid

from jsonschema import Draft202012Validator, FormatChecker

In [2]:
# =========================================================
# Project Configuration
# =========================================================

PROJECT_ROOT = Path("/content/customer_support_capstone")

SCHEMA_DIR = PROJECT_ROOT / "schemas"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"
QUARANTINE_DIR = PROJECT_ROOT / "data" / "quarantine"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
LOG_DIR = PROJECT_ROOT / "logs"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
DAG_DIR = PROJECT_ROOT / "dags"
SRC_DIR = PROJECT_ROOT / "src"

In [3]:
PROJECT_ROOT = Path("/content/customer_support_capstone")

folders = [
    "data/raw",
    "data/bronze",
    "data/silver",
    "data/gold",
    "data/quarantine",
    "artifacts/vector_db",
    "logs/audit",
    "logs/lineage",
    "dags",
    "src",
    "references",
    "outputs"
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Project initialized successfully:")
print(PROJECT_ROOT)

for folder in folders:
    print("✓", folder)

Project initialized successfully:
/content/customer_support_capstone
✓ data/raw
✓ data/bronze
✓ data/silver
✓ data/gold
✓ data/quarantine
✓ artifacts/vector_db
✓ logs/audit
✓ logs/lineage
✓ dags
✓ src
✓ references
✓ outputs


In [4]:
# التأكد من وجود مسار المشروع بعد إعادة تشغيل الخلية
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path("/content/customer_support_capstone")

SCHEMA_DIR = PROJECT_ROOT / "schemas"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

SCHEMA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)


# =========================================================
# Helper functions
# =========================================================

def generate_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"


def iso_timestamp(hours_ago: int = 0) -> str:
    timestamp = datetime.now(timezone.utc) - timedelta(hours=hours_ago)
    return timestamp.isoformat()


def save_json(path: Path, data: dict) -> None:
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)


def save_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


# =========================================================
# 1. JSON Schemas
# =========================================================

ticket_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "Customer Support Ticket",
    "type": "object",
    "required": [
        "ticket_id",
        "customer_id",
        "subject",
        "description",
        "priority",
        "status",
        "channel",
        "created_at"
    ],
    "properties": {
        "ticket_id": {
            "type": "string",
            "minLength": 5
        },
        "customer_id": {
            "type": "string",
            "minLength": 5
        },
        "subject": {
            "type": "string",
            "minLength": 5
        },
        "description": {
            "type": "string",
            "minLength": 10
        },
        "priority": {
            "type": "string",
            "enum": ["low", "medium", "high", "critical"]
        },
        "status": {
            "type": "string",
            "enum": ["open", "pending", "resolved", "closed"]
        },
        "channel": {
            "type": "string",
            "enum": ["email", "chat", "phone", "web"]
        },
        "language": {
            "type": "string",
            "enum": ["en", "ar"]
        },
        "agent_response": {
            "type": ["string", "null"]
        },
        "created_at": {
            "type": "string",
            "format": "date-time"
        },
        "updated_at": {
            "type": "string",
            "format": "date-time"
        }
    },
    "additionalProperties": False
}


chat_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "Customer Support Chat Message",
    "type": "object",
    "required": [
        "message_id",
        "conversation_id",
        "customer_id",
        "role",
        "message",
        "event_time"
    ],
    "properties": {
        "message_id": {
            "type": "string",
            "minLength": 5
        },
        "conversation_id": {
            "type": "string",
            "minLength": 5
        },
        "customer_id": {
            "type": "string",
            "minLength": 5
        },
        "role": {
            "type": "string",
            "enum": ["customer", "agent", "system"]
        },
        "message": {
            "type": "string",
            "minLength": 2
        },
        "language": {
            "type": "string",
            "enum": ["en", "ar"]
        },
        "event_time": {
            "type": "string",
            "format": "date-time"
        }
    },
    "additionalProperties": False
}


knowledge_article_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "Knowledge Base Article",
    "type": "object",
    "required": [
        "article_id",
        "title",
        "content",
        "category",
        "version",
        "status",
        "updated_at"
    ],
    "properties": {
        "article_id": {
            "type": "string",
            "minLength": 5
        },
        "title": {
            "type": "string",
            "minLength": 5
        },
        "content": {
            "type": "string",
            "minLength": 50
        },
        "category": {
            "type": "string"
        },
        "version": {
            "type": "integer",
            "minimum": 1
        },
        "status": {
            "type": "string",
            "enum": ["draft", "published", "archived"]
        },
        "tags": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },
        "updated_at": {
            "type": "string",
            "format": "date-time"
        }
    },
    "additionalProperties": False
}


save_json(SCHEMA_DIR / "ticket_schema.json", ticket_schema)
save_json(SCHEMA_DIR / "chat_schema.json", chat_schema)
save_json(
    SCHEMA_DIR / "knowledge_article_schema.json",
    knowledge_article_schema
)


# =========================================================
# 2. Knowledge base articles
# =========================================================

knowledge_templates = [
    {
        "title": "How to reset your account password",
        "category": "account_access",
        "tags": ["password", "login", "security"],
        "content": (
            "Customers can reset their password from the sign-in page by selecting "
            "'Forgot Password'. A secure reset link is sent to the registered email "
            "address. The link expires after 30 minutes. Support agents must never ask "
            "customers to provide their current password."
        )
    },
    {
        "title": "Refund and return policy",
        "category": "refunds",
        "tags": ["refund", "return", "payment"],
        "content": (
            "Customers may request a refund within 30 days of purchase when the product "
            "meets the return conditions. Approved refunds are sent to the original "
            "payment method and normally take five to ten business days to appear."
        )
    },
    {
        "title": "Delayed order investigation process",
        "category": "shipping",
        "tags": ["shipping", "delivery", "tracking"],
        "content": (
            "When an order is delayed, verify the tracking number and the latest carrier "
            "scan. If no tracking update appears for more than 72 hours, open a carrier "
            "investigation and provide the customer with a case reference."
        )
    },
    {
        "title": "Resolving duplicate charges",
        "category": "billing",
        "tags": ["billing", "duplicate", "charge"],
        "content": (
            "A duplicate charge may be a temporary authorization or a completed payment. "
            "Agents should compare transaction identifiers before issuing a refund. "
            "Temporary authorizations usually disappear within seven business days."
        )
    },
    {
        "title": "Unlocking a locked customer account",
        "category": "account_access",
        "tags": ["locked", "account", "security"],
        "content": (
            "An account may be locked after multiple failed login attempts. Customers "
            "should wait fifteen minutes before retrying or use the password reset flow. "
            "Identity verification is required before an agent manually unlocks an account."
        )
    },
    {
        "title": "Cancelling a subscription",
        "category": "subscriptions",
        "tags": ["subscription", "cancel", "renewal"],
        "content": (
            "Customers can cancel a subscription from the billing settings page. "
            "Cancellation stops future renewals but access remains available until the "
            "end of the current billing period. Previous charges are governed by the refund policy."
        )
    },
    {
        "title": "Updating a shipping address",
        "category": "shipping",
        "tags": ["address", "shipping", "order"],
        "content": (
            "A shipping address can be changed only before the order enters fulfillment. "
            "After fulfillment begins, the customer must contact the carrier or request "
            "a return after delivery. Agents must verify the order number before changes."
        )
    },
    {
        "title": "Two-factor authentication support",
        "category": "security",
        "tags": ["2fa", "authentication", "security"],
        "content": (
            "Customers who cannot access their authenticator application may use a saved "
            "recovery code. If recovery codes are unavailable, support must complete the "
            "identity verification workflow before disabling two-factor authentication."
        )
    },
    {
        "title": "API rate limit guidance",
        "category": "technical",
        "tags": ["api", "rate-limit", "developer"],
        "content": (
            "API requests are limited according to the customer's service plan. "
            "Applications receiving HTTP 429 responses should use exponential backoff "
            "and inspect the rate-limit response headers before retrying."
        )
    },
    {
        "title": "Requesting a personal data export",
        "category": "privacy",
        "tags": ["privacy", "export", "data"],
        "content": (
            "Customers may request an export of their personal data from the privacy "
            "settings page. The request requires account verification and may take up "
            "to thirty days. The download link is time-limited and must not be shared."
        )
    }
]

knowledge_articles = []

for index, template in enumerate(knowledge_templates, start=1):
    knowledge_articles.append({
        "article_id": f"KB-{index:04d}",
        "title": template["title"],
        "content": template["content"],
        "category": template["category"],
        "version": 1,
        "status": "published",
        "tags": template["tags"],
        "updated_at": iso_timestamp(index)
    })


# =========================================================
# 3. Support tickets
# =========================================================

ticket_templates = [
    {
        "subject": "Unable to reset my password",
        "description": (
            "I requested a password reset email, but I have not received the reset link."
        ),
        "response": (
            "Please check your spam folder and confirm that you used the registered email. "
            "You can request another reset link from the sign-in page."
        )
    },
    {
        "subject": "Refund request for recent order",
        "description": (
            "I purchased an item twelve days ago and would like to return it for a refund."
        ),
        "response": (
            "Your purchase is within the 30-day refund period. Please submit the return "
            "request from your order page."
        )
    },
    {
        "subject": "Order delivery is delayed",
        "description": (
            "The tracking information has not changed for four days and my order has not arrived."
        ),
        "response": (
            "We will verify the carrier tracking information and open an investigation "
            "because there has been no update for more than 72 hours."
        )
    },
    {
        "subject": "Duplicate charge on my card",
        "description": (
            "My bank statement shows two charges for the same purchase."
        ),
        "response": (
            "We will compare the two transaction identifiers to determine whether one "
            "charge is a temporary authorization."
        )
    },
    {
        "subject": "My account is locked",
        "description": (
            "I entered the wrong password several times and can no longer sign in."
        ),
        "response": (
            "Please wait fifteen minutes and retry, or use the password reset option. "
            "Manual unlocking requires identity verification."
        )
    },
    {
        "subject": "Cancel my subscription",
        "description": (
            "I want to stop the next subscription renewal."
        ),
        "response": (
            "You can cancel from the billing settings page. Access will remain active "
            "until the end of the current billing period."
        )
    },
    {
        "subject": "Change delivery address",
        "description": (
            "I entered the wrong address and need to change it before the order ships."
        ),
        "response": (
            "We can update the address if the order has not entered fulfillment. "
            "Please provide the order number."
        )
    },
    {
        "subject": "Lost access to authenticator app",
        "description": (
            "I replaced my phone and cannot access my two-factor authentication codes."
        ),
        "response": (
            "Use a saved recovery code. If none is available, we must complete identity "
            "verification before disabling two-factor authentication."
        )
    },
    {
        "subject": "API returns HTTP 429",
        "description": (
            "Our integration frequently receives HTTP 429 responses from the API."
        ),
        "response": (
            "Implement exponential backoff and inspect the rate-limit response headers "
            "before retrying requests."
        )
    },
    {
        "subject": "Request account data export",
        "description": (
            "I want to download a copy of all personal information stored in my account."
        ),
        "response": (
            "Submit the request from the privacy settings page. Account verification is "
            "required, and processing may take up to thirty days."
        )
    }
]

tickets = []

for index in range(30):
    template = ticket_templates[index % len(ticket_templates)]

    created_at = datetime.now(timezone.utc) - timedelta(hours=index + 1)
    updated_at = created_at + timedelta(minutes=random.randint(5, 120))

    # بعض الردود الضعيفة عمدًا لاستخدامها لاحقًا في Quality Gate
    if index in {7, 16, 24}:
        agent_response = random.choice([
            "OK",
            "Try again later.",
            "We cannot help."
        ])
    else:
        agent_response = template["response"]

    tickets.append({
        "ticket_id": f"TKT-{index + 1:05d}",
        "customer_id": f"CUST-{1000 + index:05d}",
        "subject": template["subject"],
        "description": template["description"],
        "priority": random.choice(["low", "medium", "medium", "high"]),
        "status": random.choice(["open", "pending", "resolved"]),
        "channel": random.choice(["email", "chat", "web"]),
        "language": "en",
        "agent_response": agent_response,
        "created_at": created_at.isoformat(),
        "updated_at": updated_at.isoformat()
    })


# =========================================================
# 4. Chat conversations
# =========================================================

chat_pairs = [
    (
        "I cannot reset my password. What should I do?",
        "Use the Forgot Password option and check the registered email address."
    ),
    (
        "Can I get a refund after twelve days?",
        "Yes. Eligible purchases can be refunded within thirty days."
    ),
    (
        "My shipment has not moved for four days.",
        "We should open a carrier investigation because the delay exceeds 72 hours."
    ),
    (
        "Why did I receive two card charges?",
        "One may be a temporary authorization. We need to compare the transactions."
    ),
    (
        "How do I cancel my subscription?",
        "Open billing settings and select Cancel Subscription."
    )
]

chat_messages = []

for conversation_index in range(15):
    customer_message, agent_message = chat_pairs[
        conversation_index % len(chat_pairs)
    ]

    conversation_id = f"CONV-{conversation_index + 1:04d}"
    customer_id = f"CUST-{2000 + conversation_index:05d}"
    base_time = datetime.now(timezone.utc) - timedelta(
        minutes=conversation_index * 10
    )

    chat_messages.append({
        "message_id": generate_id("MSG"),
        "conversation_id": conversation_id,
        "customer_id": customer_id,
        "role": "customer",
        "message": customer_message,
        "language": "en",
        "event_time": base_time.isoformat()
    })

    chat_messages.append({
        "message_id": generate_id("MSG"),
        "conversation_id": conversation_id,
        "customer_id": customer_id,
        "role": "agent",
        "message": agent_message,
        "language": "en",
        "event_time": (
            base_time + timedelta(seconds=random.randint(20, 90))
        ).isoformat()
    })


# =========================================================
# 5. Invalid records for quarantine demonstration
# =========================================================

invalid_records = [
    {
        "record_type": "ticket",
        "data": {
            "ticket_id": "BAD-001",
            "customer_id": "CUST-999",
            "subject": "Missing description",
            "priority": "high",
            "status": "open",
            "channel": "email",
            "created_at": iso_timestamp()
        }
    },
    {
        "record_type": "chat",
        "data": {
            "message_id": "BAD-MSG-001",
            "conversation_id": "CONV-BAD",
            "customer_id": "CUST-BAD",
            "role": "unknown_role",
            "message": "Test message",
            "event_time": iso_timestamp()
        }
    },
    {
        "record_type": "knowledge_article",
        "data": {
            "article_id": "BAD-KB-001",
            "title": "Empty article",
            "content": "",
            "category": "unknown",
            "version": 0,
            "status": "published",
            "updated_at": iso_timestamp()
        }
    }
]


# =========================================================
# 6. Save datasets
# =========================================================

save_jsonl(RAW_DIR / "support_tickets.jsonl", tickets)
save_jsonl(RAW_DIR / "chat_messages.jsonl", chat_messages)
save_jsonl(RAW_DIR / "knowledge_articles.jsonl", knowledge_articles)
save_jsonl(RAW_DIR / "invalid_records.jsonl", invalid_records)


# =========================================================
# 7. Summary
# =========================================================

print("Data preparation completed successfully.")
print("-" * 55)
print(f"Ticket records:            {len(tickets)}")
print(f"Chat message records:      {len(chat_messages)}")
print(f"Knowledge base articles:   {len(knowledge_articles)}")
print(f"Invalid demo records:      {len(invalid_records)}")
print("-" * 55)

print("\nSchema files:")
for file_path in sorted(SCHEMA_DIR.glob("*.json")):
    print("✓", file_path)

print("\nRaw data files:")
for file_path in sorted(RAW_DIR.glob("*.jsonl")):
    print("✓", file_path)

Data preparation completed successfully.
-------------------------------------------------------
Ticket records:            30
Chat message records:      30
Knowledge base articles:   10
Invalid demo records:      3
-------------------------------------------------------

Schema files:
✓ /content/customer_support_capstone/schemas/chat_schema.json
✓ /content/customer_support_capstone/schemas/knowledge_article_schema.json
✓ /content/customer_support_capstone/schemas/ticket_schema.json

Raw data files:
✓ /content/customer_support_capstone/data/raw/chat_messages.jsonl
✓ /content/customer_support_capstone/data/raw/invalid_records.jsonl
✓ /content/customer_support_capstone/data/raw/knowledge_articles.jsonl
✓ /content/customer_support_capstone/data/raw/support_tickets.jsonl


In [5]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re


# =========================================================
# Project paths
# =========================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path("/content/customer_support_capstone")

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

SILVER_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# Helper functions
# =========================================================

def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def save_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )


def normalize_text(value):
    if value is None:
        return None

    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)

    return value


def mask_pii(text):
    """
    إخفاء البريد الإلكتروني وأرقام الهاتف وبطاقات الدفع
    داخل النصوص قبل إدخالها إلى Silver وVector DB.
    """

    if text is None:
        return None

    text = str(text)

    # Email addresses
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "[EMAIL_REDACTED]",
        text
    )

    # Phone numbers
    text = re.sub(
        r"(?<!\d)(?:\+?\d[\d\s\-()]{7,}\d)(?!\d)",
        "[PHONE_REDACTED]",
        text
    )

    # Possible payment-card numbers
    text = re.sub(
        r"\b(?:\d[ -]*?){13,19}\b",
        "[CARD_REDACTED]",
        text
    )

    return text


def parse_timestamp(value):
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)

    try:
        return datetime.fromisoformat(
            value.replace("Z", "+00:00")
        )
    except (ValueError, TypeError):
        return datetime.min.replace(tzinfo=timezone.utc)


def deduplicate_records(
    records: list[dict],
    key_field: str,
    timestamp_field: str
) -> tuple[list[dict], int]:

    latest_records = {}

    for record in records:
        key = record.get(key_field)

        if not key:
            continue

        if key not in latest_records:
            latest_records[key] = record
            continue

        current_timestamp = parse_timestamp(
            record.get(timestamp_field)
        )

        stored_timestamp = parse_timestamp(
            latest_records[key].get(timestamp_field)
        )

        if current_timestamp >= stored_timestamp:
            latest_records[key] = record

    deduplicated = list(latest_records.values())
    removed_count = len(records) - len(deduplicated)

    return deduplicated, removed_count


# =========================================================
# Response quality evaluation
# =========================================================

STOP_WORDS = {
    "the", "a", "an", "and", "or", "to", "of", "in", "on",
    "is", "are", "was", "were", "i", "my", "your", "you",
    "it", "this", "that", "for", "from", "with", "have",
    "has", "do", "can", "cannot", "not", "be", "we"
}


GENERIC_RESPONSES = {
    "ok",
    "okay",
    "try again later",
    "we cannot help",
    "contact support",
    "please wait",
    "no",
    "yes"
}


ACTION_WORDS = {
    "check",
    "confirm",
    "submit",
    "open",
    "verify",
    "use",
    "provide",
    "implement",
    "inspect",
    "request",
    "cancel",
    "update",
    "compare",
    "wait",
    "retry",
    "complete"
}


def extract_keywords(text):
    if not text:
        return set()

    words = re.findall(r"[a-zA-Z]{3,}", text.lower())

    return {
        word
        for word in words
        if word not in STOP_WORDS
    }


def evaluate_response_quality(
    subject,
    description,
    response
):
    reasons = []

    if response is None or not response.strip():
        return {
            "response_quality_score": 0.0,
            "response_quality_label": "low",
            "is_low_quality_response": True,
            "quality_reasons": ["Missing agent response"]
        }

    normalized_response = normalize_text(response)
    response_lower = normalized_response.lower().rstrip(".!?")
    response_words = extract_keywords(normalized_response)

    score = 1.0

    if len(normalized_response) < 20:
        score -= 0.45
        reasons.append("Response is too short")

    if len(response_words) < 4:
        score -= 0.20
        reasons.append("Response lacks sufficient detail")

    if response_lower in GENERIC_RESPONSES:
        score -= 0.35
        reasons.append("Response is generic or non-actionable")

    issue_keywords = extract_keywords(
        f"{subject or ''} {description or ''}"
    )

    overlap = issue_keywords.intersection(response_words)

    if issue_keywords and not overlap:
        score -= 0.15
        reasons.append("Low relevance to the customer issue")

    if not response_words.intersection(ACTION_WORDS):
        score -= 0.10
        reasons.append("No clear next action")

    score = round(max(0.0, min(1.0, score)), 2)

    if score >= 0.75:
        label = "high"
    elif score >= 0.50:
        label = "medium"
    else:
        label = "low"

    return {
        "response_quality_score": score,
        "response_quality_label": label,
        "is_low_quality_response": label == "low",
        "quality_reasons": reasons or [
            "Response passed heuristic quality checks"
        ]
    }


# =========================================================
# Load Bronze data
# =========================================================

ticket_records = load_jsonl(
    BRONZE_DIR / "support_tickets_bronze.jsonl"
)

chat_records = load_jsonl(
    BRONZE_DIR / "chat_messages_bronze.jsonl"
)

knowledge_records = load_jsonl(
    BRONZE_DIR / "knowledge_articles_bronze.jsonl"
)


# =========================================================
# Deduplication
# =========================================================

ticket_records, removed_ticket_duplicates = deduplicate_records(
    ticket_records,
    key_field="ticket_id",
    timestamp_field="updated_at"
)

chat_records, removed_chat_duplicates = deduplicate_records(
    chat_records,
    key_field="message_id",
    timestamp_field="event_time"
)

knowledge_records, removed_article_duplicates = deduplicate_records(
    knowledge_records,
    key_field="article_id",
    timestamp_field="updated_at"
)


# =========================================================
# Transform tickets
# =========================================================

silver_run_time = datetime.now(timezone.utc).isoformat()

silver_tickets = []
low_quality_responses = []

for record in ticket_records:
    cleaned_record = dict(record)

    cleaned_record["subject"] = mask_pii(
        normalize_text(record.get("subject"))
    )

    cleaned_record["description"] = mask_pii(
        normalize_text(record.get("description"))
    )

    cleaned_record["agent_response"] = mask_pii(
        normalize_text(record.get("agent_response"))
    )

    cleaned_record["priority"] = normalize_text(
        record.get("priority")
    ).lower()

    cleaned_record["status"] = normalize_text(
        record.get("status")
    ).lower()

    cleaned_record["channel"] = normalize_text(
        record.get("channel")
    ).lower()

    quality_result = evaluate_response_quality(
        cleaned_record.get("subject"),
        cleaned_record.get("description"),
        cleaned_record.get("agent_response")
    )

    cleaned_record.update(quality_result)

    created_at = parse_timestamp(
        cleaned_record.get("created_at")
    )

    updated_at = parse_timestamp(
        cleaned_record.get("updated_at")
    )

    processing_minutes = max(
        0,
        round(
            (updated_at - created_at).total_seconds() / 60,
            2
        )
    )

    cleaned_record["processing_time_minutes"] = (
        processing_minutes
    )

    cleaned_record["_silver_metadata"] = {
        "processed_at": silver_run_time,
        "pipeline_layer": "silver",
        "pii_masking_applied": True,
        "deduplication_applied": True,
        "quality_evaluation_applied": True
    }

    silver_tickets.append(cleaned_record)

    if cleaned_record["is_low_quality_response"]:
        low_quality_responses.append({
            "ticket_id": cleaned_record["ticket_id"],
            "subject": cleaned_record["subject"],
            "agent_response": cleaned_record[
                "agent_response"
            ],
            "response_quality_score": cleaned_record[
                "response_quality_score"
            ],
            "quality_reasons": cleaned_record[
                "quality_reasons"
            ]
        })


# =========================================================
# Transform chat messages
# =========================================================

silver_chat_messages = []

for record in chat_records:
    cleaned_record = dict(record)

    cleaned_record["message"] = mask_pii(
        normalize_text(record.get("message"))
    )

    cleaned_record["role"] = normalize_text(
        record.get("role")
    ).lower()

    cleaned_record["message_word_count"] = len(
        cleaned_record["message"].split()
    )

    cleaned_record["_silver_metadata"] = {
        "processed_at": silver_run_time,
        "pipeline_layer": "silver",
        "pii_masking_applied": True,
        "deduplication_applied": True
    }

    silver_chat_messages.append(cleaned_record)


# =========================================================
# Transform knowledge articles
# =========================================================

silver_knowledge_articles = []

for record in knowledge_records:
    cleaned_record = dict(record)

    cleaned_record["title"] = normalize_text(
        record.get("title")
    )

    cleaned_record["content"] = normalize_text(
        record.get("content")
    )

    cleaned_record["category"] = normalize_text(
        record.get("category")
    ).lower()

    cleaned_record["tags"] = sorted({
        normalize_text(tag).lower()
        for tag in record.get("tags", [])
        if normalize_text(tag)
    })

    cleaned_record["content_word_count"] = len(
        cleaned_record["content"].split()
    )

    cleaned_record["search_text"] = (
        f"{cleaned_record['title']}. "
        f"{cleaned_record['content']} "
        f"Category: {cleaned_record['category']}. "
        f"Tags: {' '.join(cleaned_record['tags'])}"
    )

    cleaned_record["_silver_metadata"] = {
        "processed_at": silver_run_time,
        "pipeline_layer": "silver",
        "deduplication_applied": True,
        "ready_for_vector_index": (
            cleaned_record.get("status") == "published"
        )
    }

    silver_knowledge_articles.append(cleaned_record)


# =========================================================
# Save Silver outputs
# =========================================================

ticket_silver_path = (
    SILVER_DIR / "support_tickets_silver.jsonl"
)

chat_silver_path = (
    SILVER_DIR / "chat_messages_silver.jsonl"
)

knowledge_silver_path = (
    SILVER_DIR / "knowledge_articles_silver.jsonl"
)

low_quality_path = (
    OUTPUT_DIR / "low_quality_responses.jsonl"
)

save_jsonl(ticket_silver_path, silver_tickets)
save_jsonl(chat_silver_path, silver_chat_messages)
save_jsonl(
    knowledge_silver_path,
    silver_knowledge_articles
)
save_jsonl(low_quality_path, low_quality_responses)


# =========================================================
# Silver quality report
# =========================================================

high_quality_count = sum(
    item["response_quality_label"] == "high"
    for item in silver_tickets
)

medium_quality_count = sum(
    item["response_quality_label"] == "medium"
    for item in silver_tickets
)

low_quality_count = sum(
    item["response_quality_label"] == "low"
    for item in silver_tickets
)

silver_report = {
    "silver_run_at": silver_run_time,
    "ticket_records": len(silver_tickets),
    "chat_message_records": len(silver_chat_messages),
    "knowledge_articles": len(
        silver_knowledge_articles
    ),
    "duplicates_removed": {
        "tickets": removed_ticket_duplicates,
        "chat_messages": removed_chat_duplicates,
        "knowledge_articles": removed_article_duplicates
    },
    "response_quality": {
        "high": high_quality_count,
        "medium": medium_quality_count,
        "low": low_quality_count
    },
    "pii_masking_applied": True,
    "silver_files": {
        "tickets": str(ticket_silver_path),
        "chat_messages": str(chat_silver_path),
        "knowledge_articles": str(
            knowledge_silver_path
        )
    },
    "low_quality_output": str(low_quality_path)
}

silver_report_path = (
    OUTPUT_DIR / "silver_quality_report.json"
)

with silver_report_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        silver_report,
        file,
        ensure_ascii=False,
        indent=2
    )


# =========================================================
# Print results
# =========================================================

print("Silver transformation completed successfully.")
print("-" * 62)
print(
    f"Silver ticket records:          "
    f"{len(silver_tickets)}"
)
print(
    f"Silver chat records:            "
    f"{len(silver_chat_messages)}"
)
print(
    f"Silver knowledge articles:      "
    f"{len(silver_knowledge_articles)}"
)
print("-" * 62)
print(
    f"High-quality responses:         "
    f"{high_quality_count}"
)
print(
    f"Medium-quality responses:       "
    f"{medium_quality_count}"
)
print(
    f"Low-quality responses:          "
    f"{low_quality_count}"
)
print("-" * 62)
print(
    f"Removed ticket duplicates:      "
    f"{removed_ticket_duplicates}"
)
print(
    f"Removed chat duplicates:        "
    f"{removed_chat_duplicates}"
)
print(
    f"Removed article duplicates:     "
    f"{removed_article_duplicates}"
)

print("\nSilver files:")
print("✓", ticket_silver_path)
print("✓", chat_silver_path)
print("✓", knowledge_silver_path)

print("\nQuality outputs:")
print("✓", low_quality_path)
print("✓", silver_report_path)

print("\nFlagged low-quality responses:")

for item in low_quality_responses:
    print(
        f"\n- {item['ticket_id']}"
        f"\n  Response: {item['agent_response']}"
        f"\n  Score: {item['response_quality_score']}"
        f"\n  Reasons: {item['quality_reasons']}"
    )

In [6]:
# تثبيت jsonschema إذا لم تكن موجودة
if importlib.util.find_spec("jsonschema") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "jsonschema"
    ])

from jsonschema import Draft202012Validator, FormatChecker


# =========================================================
# Project paths
# =========================================================

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path("/content/customer_support_capstone")

SCHEMA_DIR = PROJECT_ROOT / "schemas"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
QUARANTINE_DIR = PROJECT_ROOT / "data" / "quarantine"

BRONZE_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# Helper functions
# =========================================================

def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                records.append({
                    "_json_parse_error": str(error),
                    "_line_number": line_number,
                    "_raw_line": line
                })

    return records


def save_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )


def validate_record(
    record: dict,
    validator: Draft202012Validator
) -> tuple[bool, list[str]]:

    errors = sorted(
        validator.iter_errors(record),
        key=lambda error: list(error.path)
    )

    messages = []

    for error in errors:
        field_path = ".".join(str(item) for item in error.path)

        if field_path:
            messages.append(f"{field_path}: {error.message}")
        else:
            messages.append(error.message)

    return len(messages) == 0, messages


# =========================================================
# Load schemas and create validators
# =========================================================

schemas = {
    "ticket": load_json(SCHEMA_DIR / "ticket_schema.json"),
    "chat": load_json(SCHEMA_DIR / "chat_schema.json"),
    "knowledge_article": load_json(
        SCHEMA_DIR / "knowledge_article_schema.json"
    )
}

validators = {
    record_type: Draft202012Validator(
        schema,
        format_checker=FormatChecker()
    )
    for record_type, schema in schemas.items()
}


# =========================================================
# Source configuration
# =========================================================

sources = {
    "ticket": RAW_DIR / "support_tickets.jsonl",
    "chat": RAW_DIR / "chat_messages.jsonl",
    "knowledge_article": RAW_DIR / "knowledge_articles.jsonl"
}

bronze_outputs = {
    "ticket": BRONZE_DIR / "support_tickets_bronze.jsonl",
    "chat": BRONZE_DIR / "chat_messages_bronze.jsonl",
    "knowledge_article": (
        BRONZE_DIR / "knowledge_articles_bronze.jsonl"
    )
}


# =========================================================
# Validate normal source files
# =========================================================

validation_time = datetime.now(timezone.utc).isoformat()

valid_records_by_type = {
    "ticket": [],
    "chat": [],
    "knowledge_article": []
}

quarantined_records = []

for record_type, source_path in sources.items():

    source_records = load_jsonl(source_path)
    validator = validators[record_type]

    for source_index, record in enumerate(source_records, start=1):

        # التعامل مع أخطاء JSON إن وجدت
        if "_json_parse_error" in record:
            quarantined_records.append({
                "record_type": record_type,
                "source_file": source_path.name,
                "source_index": source_index,
                "validation_status": "failed",
                "validation_errors": [
                    record["_json_parse_error"]
                ],
                "quarantined_at": validation_time,
                "original_record": record
            })
            continue

        is_valid, validation_errors = validate_record(
            record,
            validator
        )

        if is_valid:
            bronze_record = {
                **record,
                "_metadata": {
                    "record_type": record_type,
                    "source_file": source_path.name,
                    "source_index": source_index,
                    "ingested_at": validation_time,
                    "validation_status": "passed",
                    "pipeline_layer": "bronze"
                }
            }

            valid_records_by_type[record_type].append(
                bronze_record
            )

        else:
            quarantined_records.append({
                "record_type": record_type,
                "source_file": source_path.name,
                "source_index": source_index,
                "validation_status": "failed",
                "validation_errors": validation_errors,
                "quarantined_at": validation_time,
                "original_record": record
            })


# =========================================================
# Validate intentionally invalid demo records
# =========================================================

invalid_demo_path = RAW_DIR / "invalid_records.jsonl"
invalid_demo_records = load_jsonl(invalid_demo_path)

for source_index, wrapped_record in enumerate(
    invalid_demo_records,
    start=1
):
    record_type = wrapped_record.get("record_type")
    record = wrapped_record.get("data", {})

    if record_type not in validators:
        quarantined_records.append({
            "record_type": record_type or "unknown",
            "source_file": invalid_demo_path.name,
            "source_index": source_index,
            "validation_status": "failed",
            "validation_errors": [
                "Unknown or missing record_type"
            ],
            "quarantined_at": validation_time,
            "original_record": record
        })
        continue

    is_valid, validation_errors = validate_record(
        record,
        validators[record_type]
    )

    if is_valid:
        # هذا الفرع موجود للاحتياط، لكن السجلات التجريبية يفترض أن تفشل
        bronze_record = {
            **record,
            "_metadata": {
                "record_type": record_type,
                "source_file": invalid_demo_path.name,
                "source_index": source_index,
                "ingested_at": validation_time,
                "validation_status": "passed",
                "pipeline_layer": "bronze"
            }
        }

        valid_records_by_type[record_type].append(
            bronze_record
        )

    else:
        quarantined_records.append({
            "record_type": record_type,
            "source_file": invalid_demo_path.name,
            "source_index": source_index,
            "validation_status": "failed",
            "validation_errors": validation_errors,
            "quarantined_at": validation_time,
            "original_record": record
        })


# =========================================================
# Save Bronze and Quarantine outputs
# =========================================================

for record_type, records in valid_records_by_type.items():
    save_jsonl(bronze_outputs[record_type], records)

quarantine_path = (
    QUARANTINE_DIR / "schema_validation_failures.jsonl"
)

save_jsonl(quarantine_path, quarantined_records)


# =========================================================
# Validation report
# =========================================================

report = {
    "validation_run_at": validation_time,
    "valid_ticket_records": len(
        valid_records_by_type["ticket"]
    ),
    "valid_chat_records": len(
        valid_records_by_type["chat"]
    ),
    "valid_knowledge_articles": len(
        valid_records_by_type["knowledge_article"]
    ),
    "total_valid_records": sum(
        len(records)
        for records in valid_records_by_type.values()
    ),
    "total_quarantined_records": len(quarantined_records),
    "bronze_files": {
        record_type: str(path)
        for record_type, path in bronze_outputs.items()
    },
    "quarantine_file": str(quarantine_path)
}

report_path = PROJECT_ROOT / "outputs" / "validation_report.json"

with report_path.open("w", encoding="utf-8") as file:
    json.dump(report, file, ensure_ascii=False, indent=2)


# =========================================================
# Print results
# =========================================================

print("Schema validation completed successfully.")
print("-" * 60)
print(
    f"Valid ticket records:          "
    f"{report['valid_ticket_records']}"
)
print(
    f"Valid chat records:            "
    f"{report['valid_chat_records']}"
)
print(
    f"Valid knowledge articles:      "
    f"{report['valid_knowledge_articles']}"
)
print(
    f"Total valid Bronze records:    "
    f"{report['total_valid_records']}"
)
print(
    f"Total quarantined records:     "
    f"{report['total_quarantined_records']}"
)
print("-" * 60)

print("\nBronze files:")
for path in bronze_outputs.values():
    print("✓", path)

print("\nQuarantine file:")
print("✓", quarantine_path)

print("\nValidation report:")
print("✓", report_path)

print("\nSample quarantine errors:")
for item in quarantined_records[:3]:
    print(
        f"\n- Type: {item['record_type']}\n"
        f"  Errors: {item['validation_errors']}"
    )

Schema validation completed successfully.
------------------------------------------------------------
Valid ticket records:          30
Valid chat records:            30
Valid knowledge articles:      10
Total valid Bronze records:    70
Total quarantined records:     3
------------------------------------------------------------

Bronze files:
✓ /content/customer_support_capstone/data/bronze/support_tickets_bronze.jsonl
✓ /content/customer_support_capstone/data/bronze/chat_messages_bronze.jsonl
✓ /content/customer_support_capstone/data/bronze/knowledge_articles_bronze.jsonl

Quarantine file:
✓ /content/customer_support_capstone/data/quarantine/schema_validation_failures.jsonl

Validation report:
✓ /content/customer_support_capstone/outputs/validation_report.json

Sample quarantine errors:

- Type: ticket
  Errors: ["'description' is a required property"]

- Type: chat
  Errors: ["role: 'unknown_role' is not one of ['customer', 'agent', 'system']"]

- Type: knowledge_article
  Errors:

# Silver Transformation & Quality Scoring

In [7]:
# =========================================================
# Silver Transformation & Quality Scoring
# =========================================================

SILVER_DIR = PROJECT_ROOT / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)


def normalize_text(value):
    if value is None:
        return None

    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)

    return value


def mask_pii(text):
    if text is None:
        return None

    text = str(text)

    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "[EMAIL_REDACTED]",
        text
    )

    text = re.sub(
        r"(?<!\d)(?:\+?\d[\d\s\-()]{7,}\d)(?!\d)",
        "[PHONE_REDACTED]",
        text
    )

    text = re.sub(
        r"\b(?:\d[ -]*?){13,19}\b",
        "[CARD_REDACTED]",
        text
    )

    return text


def parse_timestamp(value):
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)

    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except (ValueError, TypeError):
        return datetime.min.replace(tzinfo=timezone.utc)


def deduplicate_records(records, key_field, timestamp_field):
    latest_records = {}

    for record in records:
        key = record.get(key_field)

        if not key:
            continue

        if key not in latest_records:
            latest_records[key] = record
            continue

        current_timestamp = parse_timestamp(record.get(timestamp_field))
        stored_timestamp = parse_timestamp(
            latest_records[key].get(timestamp_field)
        )

        if current_timestamp >= stored_timestamp:
            latest_records[key] = record

    deduplicated = list(latest_records.values())
    removed_count = len(records) - len(deduplicated)

    return deduplicated, removed_count


STOP_WORDS = {
    "the", "a", "an", "and", "or", "to", "of", "in", "on",
    "is", "are", "was", "were", "i", "my", "your", "you",
    "it", "this", "that", "for", "from", "with", "have",
    "has", "do", "can", "cannot", "not", "be", "we"
}

GENERIC_RESPONSES = {
    "ok",
    "okay",
    "try again later",
    "we cannot help",
    "contact support",
    "please wait",
    "no",
    "yes"
}

ACTION_WORDS = {
    "check",
    "confirm",
    "submit",
    "open",
    "verify",
    "use",
    "provide",
    "implement",
    "inspect",
    "request",
    "cancel",
    "update",
    "compare",
    "wait",
    "retry",
    "complete"
}


def extract_keywords(text):
    if not text:
        return set()

    words = re.findall(r"[a-zA-Z]{3,}", text.lower())

    return {
        word
        for word in words
        if word not in STOP_WORDS
    }


def evaluate_response_quality(subject, description, response):
    reasons = []

    if response is None or not response.strip():
        return {
            "response_quality_score": 0.0,
            "response_quality_label": "low",
            "is_low_quality_response": True,
            "quality_reasons": ["Missing agent response"]
        }

    normalized_response = normalize_text(response)
    response_lower = normalized_response.lower().rstrip(".!?")
    response_words = extract_keywords(normalized_response)

    score = 1.0

    if len(normalized_response) < 20:
        score -= 0.45
        reasons.append("Response is too short")

    if len(response_words) < 4:
        score -= 0.20
        reasons.append("Response lacks sufficient detail")

    if response_lower in GENERIC_RESPONSES:
        score -= 0.35
        reasons.append("Response is generic or non-actionable")

    issue_keywords = extract_keywords(f"{subject or ''} {description or ''}")
    overlap = issue_keywords.intersection(response_words)

    if issue_keywords and not overlap:
        score -= 0.15
        reasons.append("Low relevance to the customer issue")

    if not response_words.intersection(ACTION_WORDS):
        score -= 0.10
        reasons.append("No clear next action")

    score = round(max(0.0, min(1.0, score)), 2)

    if score >= 0.75:
        label = "high"
    elif score >= 0.50:
        label = "medium"
    else:
        label = "low"

    return {
        "response_quality_score": score,
        "response_quality_label": label,
        "is_low_quality_response": label == "low",
        "quality_reasons": reasons or [
            "Response passed heuristic quality checks"
        ]
    }


ticket_records = load_jsonl(
    BRONZE_DIR / "support_tickets_bronze.jsonl"
)

chat_records = load_jsonl(
    BRONZE_DIR / "chat_messages_bronze.jsonl"
)

knowledge_records = load_jsonl(
    BRONZE_DIR / "knowledge_articles_bronze.jsonl"
)


ticket_records, removed_ticket_duplicates = deduplicate_records(
    ticket_records,
    key_field="ticket_id",
    timestamp_field="updated_at"
)

chat_records, removed_chat_duplicates = deduplicate_records(
    chat_records,
    key_field="message_id",
    timestamp_field="event_time"
)

knowledge_records, removed_article_duplicates = deduplicate_records(
    knowledge_records,
    key_field="article_id",
    timestamp_field="updated_at"
)


silver_run_time = datetime.now(timezone.utc).isoformat()

silver_tickets = []
low_quality_responses = []

for record in ticket_records:
    cleaned_record = dict(record)

    cleaned_record["subject"] = mask_pii(
        normalize_text(record.get("subject"))
    )

    cleaned_record["description"] = mask_pii(
        normalize_text(record.get("description"))
    )

    cleaned_record["agent_response"] = mask_pii(
        normalize_text(record.get("agent_response"))
    )

    cleaned_record["priority"] = normalize_text(
        record.get("priority")
    ).lower()

    cleaned_record["status"] = normalize_text(
        record.get("status")
    ).lower()

    cleaned_record["channel"] = normalize_text(
        record.get("channel")
    ).lower()

    quality_result = evaluate_response_quality(
        cleaned_record.get("subject"),
        cleaned_record.get("description"),
        cleaned_record.get("agent_response")
    )

    cleaned_record.update(quality_result)

    created_at = parse_timestamp(cleaned_record.get("created_at"))
    updated_at = parse_timestamp(cleaned_record.get("updated_at"))

    cleaned_record["processing_time_minutes"] = max(
        0,
        round((updated_at - created_at).total_seconds() / 60, 2)
    )

    cleaned_record["_silver_metadata"] = {
        "processed_at": silver_run_time,
        "pipeline_layer": "silver",
        "pii_masking_applied": True,
        "deduplication_applied": True,
        "quality_evaluation_applied": True
    }

    silver_tickets.append(cleaned_record)

    if cleaned_record["is_low_quality_response"]:
        low_quality_responses.append({
            "ticket_id": cleaned_record["ticket_id"],
            "subject": cleaned_record["subject"],
            "agent_response": cleaned_record["agent_response"],
            "response_quality_score": cleaned_record[
                "response_quality_score"
            ],
            "quality_reasons": cleaned_record["quality_reasons"]
        })


silver_chat_messages = []

for record in chat_records:
    cleaned_record = dict(record)

    cleaned_record["message"] = mask_pii(
        normalize_text(record.get("message"))
    )

    cleaned_record["role"] = normalize_text(
        record.get("role")
    ).lower()

    cleaned_record["message_word_count"] = len(
        cleaned_record["message"].split()
    )

    cleaned_record["_silver_metadata"] = {
        "processed_at": silver_run_time,
        "pipeline_layer": "silver",
        "pii_masking_applied": True,
        "deduplication_applied": True
    }

    silver_chat_messages.append(cleaned_record)


silver_knowledge_articles = []

for record in knowledge_records:
    cleaned_record = dict(record)

    cleaned_record["title"] = normalize_text(record.get("title"))
    cleaned_record["content"] = normalize_text(record.get("content"))
    cleaned_record["category"] = normalize_text(
        record.get("category")
    ).lower()

    cleaned_record["tags"] = sorted({
        normalize_text(tag).lower()
        for tag in record.get("tags", [])
        if normalize_text(tag)
    })

    cleaned_record["content_word_count"] = len(
        cleaned_record["content"].split()
    )

    cleaned_record["search_text"] = (
        f"{cleaned_record['title']}. "
        f"{cleaned_record['content']} "
        f"Category: {cleaned_record['category']}. "
        f"Tags: {' '.join(cleaned_record['tags'])}"
    )

    cleaned_record["_silver_metadata"] = {
        "processed_at": silver_run_time,
        "pipeline_layer": "silver",
        "deduplication_applied": True,
        "ready_for_vector_index": (
            cleaned_record.get("status") == "published"
        )
    }

    silver_knowledge_articles.append(cleaned_record)


ticket_silver_path = SILVER_DIR / "support_tickets_silver.jsonl"
chat_silver_path = SILVER_DIR / "chat_messages_silver.jsonl"
knowledge_silver_path = SILVER_DIR / "knowledge_articles_silver.jsonl"
low_quality_path = OUTPUT_DIR / "low_quality_responses.jsonl"

save_jsonl(ticket_silver_path, silver_tickets)
save_jsonl(chat_silver_path, silver_chat_messages)
save_jsonl(knowledge_silver_path, silver_knowledge_articles)
save_jsonl(low_quality_path, low_quality_responses)


high_quality_count = sum(
    item["response_quality_label"] == "high"
    for item in silver_tickets
)

medium_quality_count = sum(
    item["response_quality_label"] == "medium"
    for item in silver_tickets
)

low_quality_count = sum(
    item["response_quality_label"] == "low"
    for item in silver_tickets
)

silver_report = {
    "silver_run_at": silver_run_time,
    "ticket_records": len(silver_tickets),
    "chat_message_records": len(silver_chat_messages),
    "knowledge_articles": len(silver_knowledge_articles),
    "duplicates_removed": {
        "tickets": removed_ticket_duplicates,
        "chat_messages": removed_chat_duplicates,
        "knowledge_articles": removed_article_duplicates
    },
    "response_quality": {
        "high": high_quality_count,
        "medium": medium_quality_count,
        "low": low_quality_count
    },
    "pii_masking_applied": True,
    "silver_files": {
        "tickets": str(ticket_silver_path),
        "chat_messages": str(chat_silver_path),
        "knowledge_articles": str(knowledge_silver_path)
    },
    "low_quality_output": str(low_quality_path)
}

silver_report_path = OUTPUT_DIR / "silver_quality_report.json"

with silver_report_path.open("w", encoding="utf-8") as file:
    json.dump(silver_report, file, ensure_ascii=False, indent=2)


print("Silver transformation completed successfully.")
print("-" * 62)
print(f"Silver ticket records:          {len(silver_tickets)}")
print(f"Silver chat records:            {len(silver_chat_messages)}")
print(f"Silver knowledge articles:      {len(silver_knowledge_articles)}")
print("-" * 62)
print(f"High-quality responses:         {high_quality_count}")
print(f"Medium-quality responses:       {medium_quality_count}")
print(f"Low-quality responses:          {low_quality_count}")
print("-" * 62)
print(f"Removed ticket duplicates:      {removed_ticket_duplicates}")
print(f"Removed chat duplicates:        {removed_chat_duplicates}")
print(f"Removed article duplicates:     {removed_article_duplicates}")

print("\nSilver files:")
print("✓", ticket_silver_path)
print("✓", chat_silver_path)
print("✓", knowledge_silver_path)

print("\nQuality outputs:")
print("✓", low_quality_path)
print("✓", silver_report_path)

print("\nFlagged low-quality responses:")

for item in low_quality_responses:
    print(
        f"\n- {item['ticket_id']}"
        f"\n  Response: {item['agent_response']}"
        f"\n  Score: {item['response_quality_score']}"
        f"\n  Reasons: {item['quality_reasons']}"
    )

Silver transformation completed successfully.
--------------------------------------------------------------
Silver ticket records:          30
Silver chat records:            30
Silver knowledge articles:      10
--------------------------------------------------------------
High-quality responses:         27
Medium-quality responses:       0
Low-quality responses:          3
--------------------------------------------------------------
Removed ticket duplicates:      0
Removed chat duplicates:        0
Removed article duplicates:     0

Silver files:
✓ /content/customer_support_capstone/data/silver/support_tickets_silver.jsonl
✓ /content/customer_support_capstone/data/silver/chat_messages_silver.jsonl
✓ /content/customer_support_capstone/data/silver/knowledge_articles_silver.jsonl

Quality outputs:
✓ /content/customer_support_capstone/outputs/low_quality_responses.jsonl
✓ /content/customer_support_capstone/outputs/silver_quality_report.json

Flagged low-quality responses:

- TKT-000

#Gold Layer, Analytics & Contradiction Detection

In [8]:
# =========================================================
# Gold Layer, Analytics & Contradiction Detection
# =========================================================

GOLD_DIR = PROJECT_ROOT / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# Ticket category classification
# =========================================================

CATEGORY_RULES = {
    "account_access": [
        "password",
        "login",
        "sign in",
        "locked",
        "account"
    ],
    "refunds": [
        "refund",
        "return",
        "money back"
    ],
    "shipping": [
        "shipping",
        "delivery",
        "shipment",
        "tracking",
        "address"
    ],
    "billing": [
        "billing",
        "charge",
        "payment",
        "card"
    ],
    "subscriptions": [
        "subscription",
        "renewal",
        "cancel"
    ],
    "security": [
        "authentication",
        "authenticator",
        "two-factor",
        "2fa",
        "security"
    ],
    "technical": [
        "api",
        "http",
        "rate limit",
        "integration"
    ],
    "privacy": [
        "privacy",
        "personal information",
        "data export"
    ]
}


def classify_ticket_category(subject, description):
    combined_text = (
        f"{subject or ''} {description or ''}"
    ).lower()

    category_scores = {}

    for category, keywords in CATEGORY_RULES.items():
        category_scores[category] = sum(
            keyword in combined_text
            for keyword in keywords
        )

    best_category = max(
        category_scores,
        key=category_scores.get
    )

    if category_scores[best_category] == 0:
        return "general_support"

    return best_category


# =========================================================
# SLA enrichment
# =========================================================

SLA_TARGETS_MINUTES = {
    "critical": 30,
    "high": 60,
    "medium": 120,
    "low": 240
}


gold_tickets = []

for ticket in silver_tickets:
    gold_record = dict(ticket)

    category = classify_ticket_category(
        ticket.get("subject"),
        ticket.get("description")
    )

    sla_target = SLA_TARGETS_MINUTES.get(
        ticket.get("priority"),
        240
    )

    processing_time = ticket.get(
        "processing_time_minutes",
        0
    )

    gold_record["issue_category"] = category
    gold_record["sla_target_minutes"] = sla_target
    gold_record["sla_breached"] = (
        processing_time > sla_target
    )

    gold_record["_gold_metadata"] = {
        "processed_at": datetime.now(
            timezone.utc
        ).isoformat(),
        "pipeline_layer": "gold",
        "analytics_ready": True
    }

    gold_tickets.append(gold_record)


# =========================================================
# Distribution helper
# =========================================================

def calculate_distribution(records, field_name):
    distribution = {}

    for record in records:
        value = record.get(field_name, "unknown")

        distribution[value] = (
            distribution.get(value, 0) + 1
        )

    return distribution


# =========================================================
# Gold KPI calculations
# =========================================================

total_tickets = len(gold_tickets)

average_processing_time = round(
    sum(
        ticket.get("processing_time_minutes", 0)
        for ticket in gold_tickets
    ) / total_tickets,
    2
) if total_tickets else 0


sla_breach_count = sum(
    ticket["sla_breached"]
    for ticket in gold_tickets
)

low_quality_count = sum(
    ticket["is_low_quality_response"]
    for ticket in gold_tickets
)

unique_conversations = len({
    message["conversation_id"]
    for message in silver_chat_messages
})


support_kpis = {
    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "total_tickets": total_tickets,
    "total_chat_messages": len(
        silver_chat_messages
    ),
    "total_conversations": unique_conversations,
    "published_knowledge_articles": sum(
        article.get("status") == "published"
        for article in silver_knowledge_articles
    ),
    "average_processing_time_minutes": (
        average_processing_time
    ),
    "sla_breach_count": sla_breach_count,
    "sla_compliance_rate": round(
        (
            (total_tickets - sla_breach_count)
            / total_tickets
        ) * 100,
        2
    ) if total_tickets else 0,
    "low_quality_response_count": low_quality_count,
    "low_quality_response_rate": round(
        (low_quality_count / total_tickets) * 100,
        2
    ) if total_tickets else 0,
    "priority_distribution": calculate_distribution(
        gold_tickets,
        "priority"
    ),
    "status_distribution": calculate_distribution(
        gold_tickets,
        "status"
    ),
    "channel_distribution": calculate_distribution(
        gold_tickets,
        "channel"
    ),
    "quality_distribution": calculate_distribution(
        gold_tickets,
        "response_quality_label"
    ),
    "category_distribution": calculate_distribution(
        gold_tickets,
        "issue_category"
    )
}


# =========================================================
# Knowledge corpus for RAG
# =========================================================

knowledge_corpus = []

for article in silver_knowledge_articles:
    if article.get("status") != "published":
        continue

    knowledge_corpus.append({
        "document_id": article["article_id"],
        "source_type": "knowledge_base",
        "title": article["title"],
        "content": article["content"],
        "search_text": article["search_text"],
        "category": article["category"],
        "tags": article.get("tags", []),
        "citation": (
            f"Knowledge Base Article "
            f"{article['article_id']}"
        ),
        "updated_at": article["updated_at"]
    })


for ticket in gold_tickets:
    if ticket["response_quality_label"] != "high":
        continue

    historical_content = (
        f"Customer issue: {ticket['description']} "
        f"Recommended response: "
        f"{ticket['agent_response']}"
    )

    knowledge_corpus.append({
        "document_id": (
            f"HISTORY-{ticket['ticket_id']}"
        ),
        "source_type": "historical_ticket",
        "title": ticket["subject"],
        "content": historical_content,
        "search_text": (
            f"{ticket['subject']}. "
            f"{historical_content}"
        ),
        "category": ticket["issue_category"],
        "tags": [
            ticket["priority"],
            ticket["channel"],
            ticket["status"]
        ],
        "citation": (
            f"Historical Support Ticket "
            f"{ticket['ticket_id']}"
        ),
        "updated_at": ticket["updated_at"]
    })


# =========================================================
# Contradiction detection
# =========================================================

NUMBER_WORD_MAP = {
    "seven": "7",
    "twelve": "12",
    "fifteen": "15",
    "thirty": "30",
    "seventy-two": "72",
    "seventy two": "72"
}


def extract_policy_numbers(text):
    if not text:
        return set()

    normalized_text = text.lower()
    detected_numbers = set(
        re.findall(r"\b\d+\b", normalized_text)
    )

    for number_word, normalized_number in (
        NUMBER_WORD_MAP.items()
    ):
        if number_word in normalized_text:
            detected_numbers.add(normalized_number)

    return detected_numbers


articles_by_category = {}

for article in silver_knowledge_articles:
    category = article.get("category")

    articles_by_category.setdefault(
        category,
        []
    ).append(article)


contradiction_alerts = []

for ticket in gold_tickets:
    ticket_category = ticket["issue_category"]

    matching_articles = articles_by_category.get(
        ticket_category,
        []
    )

    response_text = ticket.get(
        "agent_response",
        ""
    )

    response_numbers = extract_policy_numbers(
        response_text
    )

    for article in matching_articles:
        article_numbers = extract_policy_numbers(
            article.get("content", "")
        )

        number_conflict = (
            bool(response_numbers)
            and bool(article_numbers)
            and response_numbers.isdisjoint(
                article_numbers
            )
        )

        response_lower = response_text.lower()
        article_lower = article.get(
            "content",
            ""
        ).lower()

        negation_conflict = (
            any(
                phrase in response_lower
                for phrase in [
                    "not allowed",
                    "cannot",
                    "never available"
                ]
            )
            and any(
                phrase in article_lower
                for phrase in [
                    "can ",
                    "may ",
                    "available"
                ]
            )
        )

        if number_conflict or negation_conflict:
            contradiction_alerts.append({
                "ticket_id": ticket["ticket_id"],
                "article_id": article["article_id"],
                "category": ticket_category,
                "agent_response": response_text,
                "knowledge_statement": (
                    article["content"]
                ),
                "contradiction_type": (
                    "numeric_policy_conflict"
                    if number_conflict
                    else "negation_conflict"
                ),
                "response_numbers": sorted(
                    response_numbers
                ),
                "article_numbers": sorted(
                    article_numbers
                ),
                "detected_at": datetime.now(
                    timezone.utc
                ).isoformat()
            })


# =========================================================
# Save Gold outputs
# =========================================================

gold_ticket_path = (
    GOLD_DIR / "support_tickets_gold.jsonl"
)

knowledge_corpus_path = (
    GOLD_DIR / "knowledge_corpus_gold.jsonl"
)

contradiction_path = (
    GOLD_DIR / "contradiction_alerts.jsonl"
)

kpi_path = (
    GOLD_DIR / "support_kpis_gold.json"
)

save_jsonl(
    gold_ticket_path,
    gold_tickets
)

save_jsonl(
    knowledge_corpus_path,
    knowledge_corpus
)

save_jsonl(
    contradiction_path,
    contradiction_alerts
)

with kpi_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        support_kpis,
        file,
        ensure_ascii=False,
        indent=2
    )


# =========================================================
# Gold summary report
# =========================================================

gold_summary = {
    "gold_run_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "gold_ticket_records": len(gold_tickets),
    "rag_corpus_documents": len(
        knowledge_corpus
    ),
    "knowledge_base_documents": sum(
        document["source_type"]
        == "knowledge_base"
        for document in knowledge_corpus
    ),
    "historical_ticket_documents": sum(
        document["source_type"]
        == "historical_ticket"
        for document in knowledge_corpus
    ),
    "contradiction_alerts": len(
        contradiction_alerts
    ),
    "gold_files": {
        "tickets": str(gold_ticket_path),
        "knowledge_corpus": str(
            knowledge_corpus_path
        ),
        "contradictions": str(
            contradiction_path
        ),
        "support_kpis": str(kpi_path)
    }
}

gold_summary_path = (
    OUTPUT_DIR / "gold_summary_report.json"
)

with gold_summary_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        gold_summary,
        file,
        ensure_ascii=False,
        indent=2
    )


# =========================================================
# Results
# =========================================================

print("Gold layer completed successfully.")
print("-" * 62)
print(
    f"Gold ticket records:            "
    f"{len(gold_tickets)}"
)
print(
    f"RAG corpus documents:           "
    f"{len(knowledge_corpus)}"
)
print(
    f"Knowledge-base documents:       "
    f"{gold_summary['knowledge_base_documents']}"
)
print(
    f"Historical ticket documents:    "
    f"{gold_summary['historical_ticket_documents']}"
)
print(
    f"Contradiction alerts:           "
    f"{len(contradiction_alerts)}"
)
print("-" * 62)
print(
    f"Average processing time:        "
    f"{average_processing_time} minutes"
)
print(
    f"SLA compliance rate:            "
    f"{support_kpis['sla_compliance_rate']}%"
)
print(
    f"Low-quality response rate:      "
    f"{support_kpis['low_quality_response_rate']}%"
)

print("\nGold files:")
print("✓", gold_ticket_path)
print("✓", knowledge_corpus_path)
print("✓", contradiction_path)
print("✓", kpi_path)
print("✓", gold_summary_path)

print("\nTicket categories:")
for category, count in sorted(
    support_kpis["category_distribution"].items()
):
    print(f"- {category}: {count}")

Gold layer completed successfully.
--------------------------------------------------------------
Gold ticket records:            30
RAG corpus documents:           37
Knowledge-base documents:       10
Historical ticket documents:    27
Contradiction alerts:           2
--------------------------------------------------------------
Average processing time:        69.1 minutes
SLA compliance rate:            93.33%
Low-quality response rate:      10.0%

Gold files:
✓ /content/customer_support_capstone/data/gold/support_tickets_gold.jsonl
✓ /content/customer_support_capstone/data/gold/knowledge_corpus_gold.jsonl
✓ /content/customer_support_capstone/data/gold/contradiction_alerts.jsonl
✓ /content/customer_support_capstone/data/gold/support_kpis_gold.json
✓ /content/customer_support_capstone/outputs/gold_summary_report.json

Ticket categories:
- account_access: 6
- billing: 3
- privacy: 3
- refunds: 3
- security: 3
- shipping: 6
- subscriptions: 3
- technical: 3


# Refined Contradiction Detection

In [9]:
# =========================================================
# Refined Contradiction Detection
# =========================================================

def calculate_article_relevance(ticket, article):
    issue_text = (
        f"{ticket.get('subject', '')} "
        f"{ticket.get('description', '')}"
    )

    issue_keywords = extract_keywords(issue_text)
    title_keywords = extract_keywords(
        article.get("title", "")
    )
    content_keywords = extract_keywords(
        article.get("content", "")
    )
    tag_keywords = {
        str(tag).lower()
        for tag in article.get("tags", [])
    }

    title_overlap = issue_keywords.intersection(
        title_keywords
    )

    content_overlap = issue_keywords.intersection(
        content_keywords
    )

    tag_overlap = issue_keywords.intersection(
        tag_keywords
    )

    score = (
        3 * len(title_overlap)
        + len(content_overlap)
        + 2 * len(tag_overlap)
    )

    return score


refined_articles_by_category = {}

for article in silver_knowledge_articles:
    category = article.get("category")

    refined_articles_by_category.setdefault(
        category,
        []
    ).append(article)


refined_contradiction_alerts = []

for ticket in gold_tickets:
    ticket_category = ticket.get("issue_category")

    candidate_articles = (
        refined_articles_by_category.get(
            ticket_category,
            []
        )
    )

    if not candidate_articles:
        continue

    scored_articles = [
        (
            calculate_article_relevance(
                ticket,
                article
            ),
            article
        )
        for article in candidate_articles
    ]

    best_score, best_article = max(
        scored_articles,
        key=lambda item: item[0]
    )

    # لا نقارن السياسات إلا عند وجود تطابق موضوعي كافٍ
    if best_score < 2:
        continue

    response_text = ticket.get(
        "agent_response",
        ""
    )

    article_text = best_article.get(
        "content",
        ""
    )

    response_numbers = extract_policy_numbers(
        response_text
    )

    article_numbers = extract_policy_numbers(
        article_text
    )

    issue_keywords = extract_keywords(
        f"{ticket.get('subject', '')} "
        f"{ticket.get('description', '')}"
    )

    article_keywords = extract_keywords(
        f"{best_article.get('title', '')} "
        f"{article_text}"
    )

    shared_context = issue_keywords.intersection(
        article_keywords
    )

    numeric_conflict = (
        len(shared_context) >= 2
        and bool(response_numbers)
        and bool(article_numbers)
        and response_numbers.isdisjoint(
            article_numbers
        )
    )

    response_lower = response_text.lower()
    article_lower = article_text.lower()

    negative_response = any(
        phrase in response_lower
        for phrase in [
            "not allowed",
            "cannot be done",
            "never available",
            "not possible"
        ]
    )

    positive_policy = any(
        phrase in article_lower
        for phrase in [
            "customers can",
            "customers may",
            "is available",
            "can be changed",
            "can cancel"
        ]
    )

    negation_conflict = (
        len(shared_context) >= 2
        and negative_response
        and positive_policy
    )

    if numeric_conflict or negation_conflict:
        refined_contradiction_alerts.append({
            "ticket_id": ticket["ticket_id"],
            "article_id": best_article["article_id"],
            "category": ticket_category,
            "article_relevance_score": best_score,
            "shared_context": sorted(
                shared_context
            ),
            "agent_response": response_text,
            "knowledge_statement": article_text,
            "contradiction_type": (
                "numeric_policy_conflict"
                if numeric_conflict
                else "negation_conflict"
            ),
            "response_numbers": sorted(
                response_numbers
            ),
            "article_numbers": sorted(
                article_numbers
            ),
            "detected_at": datetime.now(
                timezone.utc
            ).isoformat()
        })


contradiction_path = (
    GOLD_DIR / "contradiction_alerts.jsonl"
)

save_jsonl(
    contradiction_path,
    refined_contradiction_alerts
)


gold_summary["contradiction_alerts"] = len(
    refined_contradiction_alerts
)

gold_summary[
    "contradiction_detection_method"
] = "best_article_context_matching"


with gold_summary_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        gold_summary,
        file,
        ensure_ascii=False,
        indent=2
    )


print("Refined contradiction detection completed.")
print("-" * 60)
print(
    f"Previous alerts:              "
    f"{len(contradiction_alerts)}"
)
print(
    f"Refined contradiction alerts: "
    f"{len(refined_contradiction_alerts)}"
)
print("-" * 60)
print("✓", contradiction_path)
print("✓", gold_summary_path)

if refined_contradiction_alerts:
    print("\nVerified contradiction alerts:")

    for alert in refined_contradiction_alerts:
        print(
            f"\n- Ticket: {alert['ticket_id']}"
            f"\n  Article: {alert['article_id']}"
            f"\n  Type: {alert['contradiction_type']}"
            f"\n  Shared context: "
            f"{alert['shared_context']}"
        )
else:
    print(
        "\nNo verified contradictions were found. "
        "The previous alerts were false positives "
        "caused by broad category matching."
    )

Refined contradiction detection completed.
------------------------------------------------------------
Previous alerts:              2
Refined contradiction alerts: 0
------------------------------------------------------------
✓ /content/customer_support_capstone/data/gold/contradiction_alerts.jsonl
✓ /content/customer_support_capstone/outputs/gold_summary_report.json

No verified contradictions were found. The previous alerts were false positives caused by broad category matching.


# RAG Pipeline: Qdrant, Hybrid Search & Reranking

In [10]:
# =========================================================
# RAG Dependencies
# =========================================================

!pip -q install qdrant-client sentence-transformers

import math
from collections import Counter

import numpy as np

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    PointStruct,
    VectorParams
)

from sentence_transformers import (
    CrossEncoder,
    SentenceTransformer
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 7.7 MB/s eta 0:00:00


In [11]:
# =========================================================
# RAG Pipeline: Chunking, Indexing and Hybrid Retrieval
# =========================================================

VECTOR_DB_DIR = PROJECT_ROOT / "artifacts" / "vector_db"
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "customer_support_knowledge"

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

RERANKER_MODEL_NAME = (
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


# =========================================================
# Document chunking
# =========================================================

def chunk_text(text, max_words=80, overlap_words=15):
    words = normalize_text(text).split()

    if not words:
        return []

    step_size = max_words - overlap_words
    chunks = []

    for start_index in range(0, len(words), step_size):
        chunk_words = words[
            start_index:start_index + max_words
        ]

        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start_index + max_words >= len(words):
            break

    return chunks


rag_chunks = []

for document in knowledge_corpus:
    document_chunks = chunk_text(
        document["content"],
        max_words=80,
        overlap_words=15
    )

    for chunk_index, chunk_content in enumerate(
        document_chunks,
        start=1
    ):
        rag_chunks.append({
            "chunk_id": (
                f"{document['document_id']}"
                f"::chunk-{chunk_index}"
            ),
            "document_id": document["document_id"],
            "source_type": document["source_type"],
            "title": document["title"],
            "content": chunk_content,
            "search_text": (
                f"{document['title']}. "
                f"{chunk_content}"
            ),
            "category": document["category"],
            "tags": document.get("tags", []),
            "citation": document["citation"],
            "updated_at": document["updated_at"],
            "chunk_index": chunk_index
        })


# =========================================================
# Dense embeddings
# =========================================================

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

chunk_texts = [
    chunk["search_text"]
    for chunk in rag_chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

vector_size = int(chunk_embeddings.shape[1])


# =========================================================
# Qdrant local vector index
# =========================================================

if "qdrant_client" in globals():
    try:
        qdrant_client.close()
    except Exception:
        pass

qdrant_client = QdrantClient(
    path=str(VECTOR_DB_DIR)
)

if qdrant_client.collection_exists(
    collection_name=COLLECTION_NAME
):
    qdrant_client.delete_collection(
        collection_name=COLLECTION_NAME
    )

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=vector_size,
        distance=Distance.COSINE
    )
)

qdrant_points = []

for point_id, (chunk, embedding) in enumerate(
    zip(rag_chunks, chunk_embeddings),
    start=1
):
    qdrant_points.append(
        PointStruct(
            id=point_id,
            vector=embedding.tolist(),
            payload=chunk
        )
    )

qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=qdrant_points,
    wait=True
)


# =========================================================
# BM25 lexical index
# =========================================================

def tokenize_for_search(text):
    return re.findall(
        r"[a-zA-Z0-9]+",
        normalize_text(text).lower()
    )


bm25_documents = [
    tokenize_for_search(chunk["search_text"])
    for chunk in rag_chunks
]

bm25_document_frequency = Counter()

for document_tokens in bm25_documents:
    for token in set(document_tokens):
        bm25_document_frequency[token] += 1


bm25_document_count = len(bm25_documents)

bm25_average_length = (
    sum(len(tokens) for tokens in bm25_documents)
    / bm25_document_count
    if bm25_document_count
    else 0
)


def bm25_search(
    query,
    top_k=10,
    k1=1.5,
    b=0.75
):
    query_tokens = tokenize_for_search(query)
    ranked_documents = []

    for document_index, document_tokens in enumerate(
        bm25_documents
    ):
        term_frequency = Counter(document_tokens)
        document_length = len(document_tokens)

        score = 0.0

        for query_token in set(query_tokens):
            document_frequency = (
                bm25_document_frequency.get(
                    query_token,
                    0
                )
            )

            if document_frequency == 0:
                continue

            inverse_document_frequency = math.log(
                1
                + (
                    bm25_document_count
                    - document_frequency
                    + 0.5
                )
                / (
                    document_frequency
                    + 0.5
                )
            )

            frequency = term_frequency.get(
                query_token,
                0
            )

            denominator = (
                frequency
                + k1
                * (
                    1
                    - b
                    + b
                    * document_length
                    / max(bm25_average_length, 1)
                )
            )

            if denominator > 0:
                score += (
                    inverse_document_frequency
                    * frequency
                    * (k1 + 1)
                    / denominator
                )

        if score > 0:
            ranked_documents.append({
                "point_id": document_index + 1,
                "score": float(score),
                "payload": rag_chunks[document_index]
            })

    return sorted(
        ranked_documents,
        key=lambda item: item["score"],
        reverse=True
    )[:top_k]


# =========================================================
# Qdrant dense search
# =========================================================

def dense_search(query, top_k=10):
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    query_response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=top_k,
        with_payload=True
    )

    return [
        {
            "point_id": point.id,
            "score": float(point.score),
            "payload": dict(point.payload)
        }
        for point in query_response.points
    ]


# =========================================================
# Reciprocal Rank Fusion
# =========================================================

def reciprocal_rank_fusion(
    dense_results,
    lexical_results,
    fusion_constant=60
):
    fused_results = {}

    result_groups = [
        ("dense", dense_results),
        ("bm25", lexical_results)
    ]

    for source_name, results in result_groups:
        for rank, result in enumerate(
            results,
            start=1
        ):
            chunk_id = result["payload"]["chunk_id"]

            if chunk_id not in fused_results:
                fused_results[chunk_id] = {
                    "payload": result["payload"],
                    "rrf_score": 0.0,
                    "dense_score": None,
                    "bm25_score": None
                }

            fused_results[chunk_id]["rrf_score"] += (
                1 / (fusion_constant + rank)
            )

            fused_results[chunk_id][
                f"{source_name}_score"
            ] = result["score"]

    return sorted(
        fused_results.values(),
        key=lambda item: item["rrf_score"],
        reverse=True
    )


# =========================================================
# Cross-encoder reranker
# =========================================================

reranker_model = CrossEncoder(
    RERANKER_MODEL_NAME
)


def hybrid_search(
    query,
    dense_top_k=12,
    bm25_top_k=12,
    rerank_top_k=10,
    final_top_k=5
):
    dense_results = dense_search(
        query,
        top_k=dense_top_k
    )

    lexical_results = bm25_search(
        query,
        top_k=bm25_top_k
    )

    fused_results = reciprocal_rank_fusion(
        dense_results,
        lexical_results
    )

    rerank_candidates = fused_results[
        :rerank_top_k
    ]

    if not rerank_candidates:
        return []

    reranking_pairs = [
        (
            query,
            candidate["payload"]["search_text"]
        )
        for candidate in rerank_candidates
    ]

    reranking_scores = reranker_model.predict(
        reranking_pairs
    )

    for candidate, reranking_score in zip(
        rerank_candidates,
        reranking_scores
    ):
        candidate["reranker_score"] = float(
            np.asarray(reranking_score).squeeze()
        )

    return sorted(
        rerank_candidates,
        key=lambda item: item["reranker_score"],
        reverse=True
    )[:final_top_k]


# =========================================================
# Grounded answer with citations
# =========================================================

def compose_grounded_answer(
    query,
    search_results,
    maximum_sources=3
):
    if not search_results:
        return {
            "question": query,
            "suggested_answer": (
                "No sufficiently relevant evidence "
                "was found in the indexed sources."
            ),
            "citations": [],
            "evidence": []
        }

    preferred_result = next(
        (
            result
            for result in search_results
            if result["payload"]["source_type"]
            == "knowledge_base"
        ),
        search_results[0]
    )

    suggested_answer = preferred_result[
        "payload"
    ]["content"]

    citations = []
    evidence = []

    for result in search_results[:maximum_sources]:
        citation = result["payload"]["citation"]

        if citation not in citations:
            citations.append(citation)

        evidence.append({
            "citation": citation,
            "chunk_id": result["payload"]["chunk_id"],
            "title": result["payload"]["title"],
            "source_type": result[
                "payload"
            ]["source_type"],
            "reranker_score": round(
                result["reranker_score"],
                4
            ),
            "text": result["payload"]["content"]
        })

    return {
        "question": query,
        "suggested_answer": suggested_answer,
        "citations": citations,
        "evidence": evidence
    }


# =========================================================
# Demonstration query
# =========================================================

demo_query = (
    "My order tracking has not updated for four days. "
    "What should I do?"
)

demo_search_results = hybrid_search(
    demo_query
)

demo_answer = compose_grounded_answer(
    demo_query,
    demo_search_results
)


# =========================================================
# Save RAG artifacts
# =========================================================

rag_manifest = {
    "created_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "reranker_model": RERANKER_MODEL_NAME,
    "vector_dimension": vector_size,
    "source_documents": len(knowledge_corpus),
    "indexed_chunks": len(rag_chunks),
    "retrieval_methods": [
        "dense_qdrant",
        "bm25",
        "reciprocal_rank_fusion",
        "cross_encoder_reranking"
    ]
}

rag_manifest_path = (
    OUTPUT_DIR / "rag_index_manifest.json"
)

rag_demo_path = (
    OUTPUT_DIR / "rag_demo_result.json"
)

with rag_manifest_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        rag_manifest,
        file,
        ensure_ascii=False,
        indent=2
    )

with rag_demo_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        demo_answer,
        file,
        ensure_ascii=False,
        indent=2
    )


# =========================================================
# Results
# =========================================================

print("RAG pipeline completed successfully.")
print("-" * 65)
print(
    f"Source documents:               "
    f"{len(knowledge_corpus)}"
)
print(
    f"Indexed chunks:                 "
    f"{len(rag_chunks)}"
)
print(
    f"Embedding dimension:            "
    f"{vector_size}"
)
print(
    f"Qdrant collection:              "
    f"{COLLECTION_NAME}"
)
print("-" * 65)

print("\nQuestion:")
print(demo_answer["question"])

print("\nSuggested answer:")
print(demo_answer["suggested_answer"])

print("\nCitations:")
for citation in demo_answer["citations"]:
    print("✓", citation)

print("\nTop reranked evidence:")

for rank, result in enumerate(
    demo_search_results,
    start=1
):
    print(
        f"\n{rank}. {result['payload']['title']}"
        f"\n   Source: "
        f"{result['payload']['citation']}"
        f"\n   Reranker score: "
        f"{result['reranker_score']:.4f}"
        f"\n   Dense score: "
        f"{result['dense_score']}"
        f"\n   BM25 score: "
        f"{result['bm25_score']}"
    )

print("\nSaved artifacts:")
print("✓", rag_manifest_path)
print("✓", rag_demo_path)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

RAG pipeline completed successfully.
-----------------------------------------------------------------
Source documents:               37
Indexed chunks:                 37
Embedding dimension:            384
Qdrant collection:              customer_support_knowledge
-----------------------------------------------------------------

Question:
My order tracking has not updated for four days. What should I do?

Suggested answer:
When an order is delayed, verify the tracking number and the latest carrier scan. If no tracking update appears for more than 72 hours, open a carrier investigation and provide the customer with a case reference.

Citations:
✓ Historical Support Ticket TKT-00023
✓ Historical Support Ticket TKT-00003
✓ Historical Support Ticket TKT-00013

Top reranked evidence:

1. Order delivery is delayed
   Source: Historical Support Ticket TKT-00023
   Reranker score: 7.0682
   Dense score: 0.7561166533863279
   BM25 score: 15.316906501095973

2. Order delivery is delayed
   S

# RAG Result Deduplication & Citation Correction

In [12]:
# =========================================================
# RAG Result Deduplication & Citation Correction
# =========================================================

def result_content_signature(result):
    payload = result["payload"]

    return normalize_text(
        f"{payload.get('title', '')} "
        f"{payload.get('content', '')}"
    ).lower()


def hybrid_search(
    query,
    dense_top_k=20,
    bm25_top_k=20,
    rerank_top_k=20,
    final_top_k=5,
    minimum_reranker_score=0.0
):
    dense_results = dense_search(
        query,
        top_k=dense_top_k
    )

    lexical_results = bm25_search(
        query,
        top_k=bm25_top_k
    )

    fused_results = reciprocal_rank_fusion(
        dense_results,
        lexical_results
    )

    rerank_candidates = fused_results[
        :rerank_top_k
    ]

    if not rerank_candidates:
        return []

    reranking_pairs = [
        (
            query,
            candidate["payload"]["search_text"]
        )
        for candidate in rerank_candidates
    ]

    reranking_scores = reranker_model.predict(
        reranking_pairs
    )

    for candidate, reranking_score in zip(
        rerank_candidates,
        reranking_scores
    ):
        candidate["reranker_score"] = float(
            np.asarray(reranking_score).squeeze()
        )

    ranked_candidates = sorted(
        rerank_candidates,
        key=lambda item: item["reranker_score"],
        reverse=True
    )

    unique_results = []
    seen_content = set()

    for result in ranked_candidates:
        if result["reranker_score"] < minimum_reranker_score:
            continue

        signature = result_content_signature(result)

        if signature in seen_content:
            continue

        seen_content.add(signature)
        unique_results.append(result)

        if len(unique_results) >= final_top_k:
            break

    # احتياط في حال كانت جميع الدرجات سالبة
    if not unique_results and ranked_candidates:
        unique_results.append(ranked_candidates[0])

    return unique_results


def compose_grounded_answer(
    query,
    search_results,
    maximum_sources=3
):
    if not search_results:
        return {
            "question": query,
            "suggested_answer": (
                "No sufficiently relevant evidence "
                "was found in the indexed sources."
            ),
            "primary_source": None,
            "citations": [],
            "evidence": []
        }

    preferred_result = next(
        (
            result
            for result in search_results
            if result["payload"]["source_type"]
            == "knowledge_base"
        ),
        search_results[0]
    )

    preferred_citation = preferred_result[
        "payload"
    ]["citation"]

    ordered_results = [
        preferred_result,
        *[
            result
            for result in search_results
            if result["payload"]["chunk_id"]
            != preferred_result["payload"]["chunk_id"]
        ]
    ]

    citations = []
    evidence = []

    for result in ordered_results:
        citation = result["payload"]["citation"]

        if citation not in citations:
            citations.append(citation)

        if len(evidence) < maximum_sources:
            evidence.append({
                "citation": citation,
                "chunk_id": result["payload"]["chunk_id"],
                "title": result["payload"]["title"],
                "source_type": result[
                    "payload"
                ]["source_type"],
                "reranker_score": round(
                    result["reranker_score"],
                    4
                ),
                "text": result["payload"]["content"]
            })

        if len(citations) >= maximum_sources:
            break

    return {
        "question": query,
        "suggested_answer": preferred_result[
            "payload"
        ]["content"],
        "primary_source": preferred_citation,
        "citations": citations,
        "evidence": evidence
    }


refined_demo_results = hybrid_search(
    demo_query
)

refined_demo_answer = compose_grounded_answer(
    demo_query,
    refined_demo_results
)


with rag_demo_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        refined_demo_answer,
        file,
        ensure_ascii=False,
        indent=2
    )


print("RAG result refinement completed successfully.")
print("-" * 65)

print("\nQuestion:")
print(refined_demo_answer["question"])

print("\nSuggested answer:")
print(refined_demo_answer["suggested_answer"])

print("\nPrimary source:")
print("✓", refined_demo_answer["primary_source"])

print("\nCitations:")
for citation in refined_demo_answer["citations"]:
    print("✓", citation)

print("\nDeduplicated reranked evidence:")

for rank, result in enumerate(
    refined_demo_results,
    start=1
):
    print(
        f"\n{rank}. {result['payload']['title']}"
        f"\n   Source: "
        f"{result['payload']['citation']}"
        f"\n   Source type: "
        f"{result['payload']['source_type']}"
        f"\n   Reranker score: "
        f"{result['reranker_score']:.4f}"
    )

print("\nUpdated artifact:")
print("✓", rag_demo_path)

RAG result refinement completed successfully.
-----------------------------------------------------------------

Question:
My order tracking has not updated for four days. What should I do?

Suggested answer:
When an order is delayed, verify the tracking number and the latest carrier scan. If no tracking update appears for more than 72 hours, open a carrier investigation and provide the customer with a case reference.

Primary source:
✓ Knowledge Base Article KB-0003

Citations:
✓ Knowledge Base Article KB-0003
✓ Historical Support Ticket TKT-00023

Deduplicated reranked evidence:

1. Order delivery is delayed
   Source: Historical Support Ticket TKT-00023
   Source type: historical_ticket
   Reranker score: 7.0682

2. Delayed order investigation process
   Source: Knowledge Base Article KB-0003
   Source type: knowledge_base
   Reranker score: 3.3652

Updated artifact:
✓ /content/customer_support_capstone/outputs/rag_demo_result.json


# Great Expectations Quality Gate

In [13]:
# =========================================================
# Great Expectations Dependencies
# =========================================================

!pip -q install great_expectations==1.18.2

import pandas as pd
import great_expectations as gx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.5 MB/s eta 0:00:00


In [14]:
# =========================================================
# Great Expectations Quality Gate
# =========================================================

GX_OUTPUT_DIR = OUTPUT_DIR / "great_expectations"
GX_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gx_context = gx.get_context(mode="ephemeral")

gx_run_id = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")


# =========================================================
# Convert pipeline datasets to DataFrames
# =========================================================

tickets_dataframe = pd.DataFrame(gold_tickets)

chat_dataframe = pd.DataFrame(
    silver_chat_messages
)

knowledge_dataframe = pd.DataFrame(
    silver_knowledge_articles
)


# =========================================================
# Result serialization
# =========================================================

def serialize_gx_result(validation_result):
    if hasattr(validation_result, "to_json_dict"):
        return validation_result.to_json_dict()

    try:
        return json.loads(str(validation_result))
    except (TypeError, json.JSONDecodeError):
        return {
            "success": bool(
                getattr(validation_result, "success", False)
            ),
            "raw_result": str(validation_result)
        }


# =========================================================
# Reusable suite runner
# =========================================================

def run_expectation_suite(
    dataframe,
    suite_label,
    expectations
):
    data_source_name = (
        f"{suite_label}_source_{gx_run_id}"
    )

    data_asset_name = (
        f"{suite_label}_asset_{gx_run_id}"
    )

    batch_definition_name = (
        f"{suite_label}_batch_{gx_run_id}"
    )

    suite_name = (
        f"{suite_label}_suite_{gx_run_id}"
    )

    validation_name = (
        f"{suite_label}_validation_{gx_run_id}"
    )

    data_source = gx_context.data_sources.add_pandas(
        name=data_source_name
    )

    data_asset = data_source.add_dataframe_asset(
        name=data_asset_name
    )

    batch_definition = (
        data_asset
        .add_batch_definition_whole_dataframe(
            batch_definition_name
        )
    )

    expectation_suite = gx_context.suites.add(
        gx.core.expectation_suite.ExpectationSuite(
            name=suite_name
        )
    )

    for expectation in expectations:
        expectation_suite.add_expectation(
            expectation
        )

    validation_definition = (
        gx_context.validation_definitions.add(
            gx.core.validation_definition
            .ValidationDefinition(
                name=validation_name,
                data=batch_definition,
                suite=expectation_suite
            )
        )
    )

    validation_result = (
        validation_definition.run(
            batch_parameters={
                "dataframe": dataframe
            }
        )
    )

    result_dictionary = serialize_gx_result(
        validation_result
    )

    return {
        "suite_name": suite_name,
        "row_count": len(dataframe),
        "success": bool(
            result_dictionary.get(
                "success",
                False
            )
        ),
        "result": result_dictionary
    }


# =========================================================
# Ticket expectations
# =========================================================

ticket_expectations = [
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=1,
        max_value=100000
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="ticket_id"
    ),

    gx.expectations.ExpectColumnValuesToBeUnique(
        column="ticket_id"
    ),

    gx.expectations.ExpectColumnValuesToBeInSet(
        column="priority",
        value_set=[
            "low",
            "medium",
            "high",
            "critical"
        ]
    ),

    gx.expectations.ExpectColumnValuesToBeInSet(
        column="status",
        value_set=[
            "open",
            "pending",
            "resolved",
            "closed"
        ]
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="response_quality_score",
        min_value=0.0,
        max_value=1.0
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="processing_time_minutes",
        min_value=0.0
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="issue_category"
    ),

    gx.expectations.ExpectColumnValuesToBeInSet(
        column="is_low_quality_response",
        value_set=[False],
        mostly=0.90
    )
]


# =========================================================
# Chat expectations
# =========================================================

chat_expectations = [
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=1,
        max_value=100000
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="message_id"
    ),

    gx.expectations.ExpectColumnValuesToBeUnique(
        column="message_id"
    ),

    gx.expectations.ExpectColumnValuesToBeInSet(
        column="role",
        value_set=[
            "customer",
            "agent",
            "system"
        ]
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="message_word_count",
        min_value=1
    )
]


# =========================================================
# Knowledge-base expectations
# =========================================================

knowledge_expectations = [
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=1,
        max_value=100000
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="article_id"
    ),

    gx.expectations.ExpectColumnValuesToBeUnique(
        column="article_id"
    ),

    gx.expectations.ExpectColumnValuesToBeInSet(
        column="status",
        value_set=["published"]
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="content_word_count",
        min_value=10
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="search_text"
    )
]


# =========================================================
# Run validation suites
# =========================================================

ticket_validation = run_expectation_suite(
    tickets_dataframe,
    suite_label="support_tickets",
    expectations=ticket_expectations
)

chat_validation = run_expectation_suite(
    chat_dataframe,
    suite_label="chat_messages",
    expectations=chat_expectations
)

knowledge_validation = run_expectation_suite(
    knowledge_dataframe,
    suite_label="knowledge_articles",
    expectations=knowledge_expectations
)

validation_suites = [
    ticket_validation,
    chat_validation,
    knowledge_validation
]

quality_gate_passed = all(
    validation["success"]
    for validation in validation_suites
)


# =========================================================
# Summarize individual expectation results
# =========================================================

def summarize_expectations(validation):
    expectation_results = validation[
        "result"
    ].get("results", [])

    summary = []

    for result in expectation_results:
        expectation_config = result.get(
            "expectation_config",
            {}
        )

        expectation_type = (
            expectation_config.get("type")
            or expectation_config.get(
                "expectation_type"
            )
            or "unknown_expectation"
        )

        result_details = result.get(
            "result",
            {}
        )

        summary.append({
            "expectation": expectation_type,
            "success": bool(
                result.get("success", False)
            ),
            "element_count": result_details.get(
                "element_count"
            ),
            "unexpected_count": (
                result_details.get(
                    "unexpected_count"
                )
            ),
            "unexpected_percent": (
                result_details.get(
                    "unexpected_percent"
                )
            )
        })

    return summary


quality_gate_report = {
    "validation_run_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "great_expectations_version": gx.__version__,
    "quality_gate_passed": quality_gate_passed,
    "suites": {
        validation["suite_name"]: {
            "success": validation["success"],
            "row_count": validation["row_count"],
            "expectations": summarize_expectations(
                validation
            )
        }
        for validation in validation_suites
    }
}


# =========================================================
# Save validation outputs
# =========================================================

quality_gate_report_path = (
    GX_OUTPUT_DIR
    / "great_expectations_quality_gate.json"
)

with quality_gate_report_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        quality_gate_report,
        file,
        ensure_ascii=False,
        indent=2
    )


for validation in validation_suites:
    suite_result_path = (
        GX_OUTPUT_DIR
        / f"{validation['suite_name']}_result.json"
    )

    with suite_result_path.open(
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            validation["result"],
            file,
            ensure_ascii=False,
            indent=2
        )


# =========================================================
# Results
# =========================================================

print("Great Expectations validation completed.")
print("-" * 65)
print(
    f"GX Core version:                "
    f"{gx.__version__}"
)
print(
    f"Overall quality gate:           "
    f"{'PASSED' if quality_gate_passed else 'FAILED'}"
)
print("-" * 65)

for validation in validation_suites:
    print(
        f"{validation['suite_name']}: "
        f"{'PASSED' if validation['success'] else 'FAILED'} "
        f"({validation['row_count']} rows)"
    )

    for expectation in summarize_expectations(
        validation
    ):
        status = (
            "PASS"
            if expectation["success"]
            else "FAIL"
        )

        print(
            f"  [{status}] "
            f"{expectation['expectation']}"
        )

print("\nSaved Great Expectations report:")
print("✓", quality_gate_report_path)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpm4s4jpnl' for ephemeral docs site


Calculating Metrics:   0%|          | 0/56 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/30 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/35 [00:00<?, ?it/s]

Great Expectations validation completed.
-----------------------------------------------------------------
GX Core version:                1.18.2
Overall quality gate:           PASSED
-----------------------------------------------------------------
support_tickets_suite_20260706_225457: PASSED (30 rows)
  [PASS] expect_table_row_count_to_be_between
  [PASS] expect_column_values_to_not_be_null
  [PASS] expect_column_values_to_be_unique
  [PASS] expect_column_values_to_be_in_set
  [PASS] expect_column_values_to_be_in_set
  [PASS] expect_column_values_to_be_between
  [PASS] expect_column_values_to_be_between
  [PASS] expect_column_values_to_not_be_null
  [PASS] expect_column_values_to_be_in_set
chat_messages_suite_20260706_225457: PASSED (30 rows)
  [PASS] expect_table_row_count_to_be_between
  [PASS] expect_column_values_to_not_be_null
  [PASS] expect_column_values_to_be_unique
  [PASS] expect_column_values_to_be_in_set
  [PASS] expect_column_values_to_be_between
knowledge_articles_sui

# OpenLineage Events & Audit Logging

In [15]:
# =========================================================
# OpenLineage Dependency
# =========================================================

!pip -q install openlineage-python==1.50.0

from openlineage.client import OpenLineageClient
from openlineage.client.event_v2 import (
    Dataset,
    Job,
    Run,
    RunEvent,
    RunState
)
from openlineage.client.transport.file import (
    FileConfig,
    FileTransport
)
from openlineage.client.uuid import generate_new_uuid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 3.9 MB/s eta 0:00:00


In [16]:
# =========================================================
# OpenLineage Events & Audit Logging
# =========================================================

LINEAGE_DIR = PROJECT_ROOT / "logs" / "lineage"
AUDIT_DIR = PROJECT_ROOT / "logs" / "audit"

LINEAGE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

LINEAGE_FILE = (
    LINEAGE_DIR / "openlineage_events.jsonl"
)

AUDIT_FILE = (
    AUDIT_DIR / "rag_query_audit.jsonl"
)

LINEAGE_NAMESPACE = "customer_support_capstone"
LINEAGE_PRODUCER = (
    "https://github.com/customer-support-capstone"
)


# =========================================================
# Reset lineage file for reproducible notebook execution
# =========================================================

if LINEAGE_FILE.exists():
    LINEAGE_FILE.unlink()


# =========================================================
# OpenLineage file transport
# =========================================================

lineage_file_config = FileConfig(
    log_file_path=str(LINEAGE_FILE),
    append=True
)

lineage_transport = FileTransport(
    lineage_file_config
)

lineage_client = OpenLineageClient(
    transport=lineage_transport
)


# =========================================================
# Dataset helpers
# =========================================================

def create_file_dataset(path):
    path = Path(path)

    try:
        relative_name = str(
            path.relative_to(PROJECT_ROOT)
        )
    except ValueError:
        relative_name = str(path)

    return Dataset(
        namespace=(
            f"file://{PROJECT_ROOT}"
        ),
        name=relative_name
    )


def create_logical_dataset(namespace, name):
    return Dataset(
        namespace=namespace,
        name=name
    )


# =========================================================
# OpenLineage event emitter
# =========================================================

def emit_lineage_run(
    job_name,
    input_datasets,
    output_datasets
):
    run_id = str(generate_new_uuid())

    lineage_run = Run(
        runId=run_id
    )

    lineage_job = Job(
        namespace=LINEAGE_NAMESPACE,
        name=job_name
    )

    start_event = RunEvent(
        eventType=RunState.START,
        eventTime=datetime.now(
            timezone.utc
        ).isoformat(),
        run=lineage_run,
        job=lineage_job,
        producer=LINEAGE_PRODUCER,
        inputs=input_datasets,
        outputs=output_datasets
    )

    lineage_client.emit(start_event)

    complete_event = RunEvent(
        eventType=RunState.COMPLETE,
        eventTime=datetime.now(
            timezone.utc
        ).isoformat(),
        run=lineage_run,
        job=lineage_job,
        producer=LINEAGE_PRODUCER,
        inputs=input_datasets,
        outputs=output_datasets
    )

    lineage_client.emit(complete_event)

    return {
        "job_name": job_name,
        "run_id": run_id,
        "input_count": len(input_datasets),
        "output_count": len(output_datasets)
    }


# =========================================================
# 1. Schema validation lineage
# =========================================================

schema_validation_lineage = emit_lineage_run(
    job_name="schema_validation_and_bronze_ingestion",

    input_datasets=[
        create_file_dataset(
            RAW_DIR / "support_tickets.jsonl"
        ),
        create_file_dataset(
            RAW_DIR / "chat_messages.jsonl"
        ),
        create_file_dataset(
            RAW_DIR / "knowledge_articles.jsonl"
        ),
        create_file_dataset(
            RAW_DIR / "invalid_records.jsonl"
        )
    ],

    output_datasets=[
        create_file_dataset(
            BRONZE_DIR
            / "support_tickets_bronze.jsonl"
        ),
        create_file_dataset(
            BRONZE_DIR
            / "chat_messages_bronze.jsonl"
        ),
        create_file_dataset(
            BRONZE_DIR
            / "knowledge_articles_bronze.jsonl"
        ),
        create_file_dataset(
            QUARANTINE_DIR
            / "schema_validation_failures.jsonl"
        )
    ]
)


# =========================================================
# 2. Silver transformation lineage
# =========================================================

silver_lineage = emit_lineage_run(
    job_name="silver_cleaning_and_quality_scoring",

    input_datasets=[
        create_file_dataset(
            BRONZE_DIR
            / "support_tickets_bronze.jsonl"
        ),
        create_file_dataset(
            BRONZE_DIR
            / "chat_messages_bronze.jsonl"
        ),
        create_file_dataset(
            BRONZE_DIR
            / "knowledge_articles_bronze.jsonl"
        )
    ],

    output_datasets=[
        create_file_dataset(
            SILVER_DIR
            / "support_tickets_silver.jsonl"
        ),
        create_file_dataset(
            SILVER_DIR
            / "chat_messages_silver.jsonl"
        ),
        create_file_dataset(
            SILVER_DIR
            / "knowledge_articles_silver.jsonl"
        ),
        create_file_dataset(
            OUTPUT_DIR
            / "low_quality_responses.jsonl"
        )
    ]
)


# =========================================================
# 3. Gold aggregation lineage
# =========================================================

gold_lineage = emit_lineage_run(
    job_name="gold_analytics_and_knowledge_corpus",

    input_datasets=[
        create_file_dataset(
            SILVER_DIR
            / "support_tickets_silver.jsonl"
        ),
        create_file_dataset(
            SILVER_DIR
            / "chat_messages_silver.jsonl"
        ),
        create_file_dataset(
            SILVER_DIR
            / "knowledge_articles_silver.jsonl"
        )
    ],

    output_datasets=[
        create_file_dataset(
            GOLD_DIR
            / "support_tickets_gold.jsonl"
        ),
        create_file_dataset(
            GOLD_DIR
            / "knowledge_corpus_gold.jsonl"
        ),
        create_file_dataset(
            GOLD_DIR
            / "contradiction_alerts.jsonl"
        ),
        create_file_dataset(
            GOLD_DIR
            / "support_kpis_gold.json"
        )
    ]
)


# =========================================================
# 4. RAG indexing lineage
# =========================================================

rag_lineage = emit_lineage_run(
    job_name="qdrant_hybrid_rag_indexing",

    input_datasets=[
        create_file_dataset(
            GOLD_DIR
            / "knowledge_corpus_gold.jsonl"
        )
    ],

    output_datasets=[
        create_logical_dataset(
            namespace="qdrant://local",
            name=COLLECTION_NAME
        ),
        create_file_dataset(
            OUTPUT_DIR
            / "rag_index_manifest.json"
        ),
        create_file_dataset(
            OUTPUT_DIR
            / "rag_demo_result.json"
        )
    ]
)


# =========================================================
# 5. Great Expectations lineage
# =========================================================

quality_gate_lineage = emit_lineage_run(
    job_name="great_expectations_quality_gate",

    input_datasets=[
        create_file_dataset(
            GOLD_DIR
            / "support_tickets_gold.jsonl"
        ),
        create_file_dataset(
            SILVER_DIR
            / "chat_messages_silver.jsonl"
        ),
        create_file_dataset(
            SILVER_DIR
            / "knowledge_articles_silver.jsonl"
        )
    ],

    output_datasets=[
        create_file_dataset(
            GX_OUTPUT_DIR
            / "great_expectations_quality_gate.json"
        )
    ]
)


# =========================================================
# Collect lineage run metadata
# =========================================================

lineage_runs = [
    schema_validation_lineage,
    silver_lineage,
    gold_lineage,
    rag_lineage,
    quality_gate_lineage
]


# =========================================================
# Validate saved OpenLineage events
# =========================================================

lineage_events = []

with LINEAGE_FILE.open(
    "r",
    encoding="utf-8"
) as file:
    for line in file:
        line = line.strip()

        if line:
            lineage_events.append(
                json.loads(line)
            )


start_event_count = sum(
    event.get("eventType") == "START"
    for event in lineage_events
)

complete_event_count = sum(
    event.get("eventType") == "COMPLETE"
    for event in lineage_events
)


# =========================================================
# RAG query audit logger
# =========================================================

def write_rag_audit_event(
    question,
    answer,
    search_results,
    audit_path=AUDIT_FILE
):
    retrieved_evidence = []

    for rank, result in enumerate(
        search_results,
        start=1
    ):
        payload = result["payload"]

        retrieved_evidence.append({
            "rank": rank,
            "chunk_id": payload["chunk_id"],
            "document_id": payload["document_id"],
            "source_type": payload["source_type"],
            "citation": payload["citation"],
            "reranker_score": round(
                float(result["reranker_score"]),
                6
            ),
            "dense_score": (
                round(
                    float(result["dense_score"]),
                    6
                )
                if result.get("dense_score")
                is not None
                else None
            ),
            "bm25_score": (
                round(
                    float(result["bm25_score"]),
                    6
                )
                if result.get("bm25_score")
                is not None
                else None
            )
        })

    audit_record = {
        "audit_id": str(generate_new_uuid()),
        "event_type": "rag_query",
        "event_time": datetime.now(
            timezone.utc
        ).isoformat(),
        "question": question,
        "suggested_answer": answer.get(
            "suggested_answer"
        ),
        "primary_source": answer.get(
            "primary_source"
        ),
        "citations": answer.get(
            "citations",
            []
        ),
        "retrieved_result_count": len(
            search_results
        ),
        "retrieval_pipeline": [
            "Qdrant dense search",
            "BM25 lexical search",
            "Reciprocal Rank Fusion",
            "Cross-encoder reranking"
        ],
        "embedding_model": (
            EMBEDDING_MODEL_NAME
        ),
        "reranker_model": (
            RERANKER_MODEL_NAME
        ),
        "quality_gate_passed": (
            quality_gate_passed
        ),
        "evidence": retrieved_evidence,
        "status": "success"
    }

    with audit_path.open(
        "a",
        encoding="utf-8"
    ) as file:
        file.write(
            json.dumps(
                audit_record,
                ensure_ascii=False
            )
            + "\n"
        )

    return audit_record


# =========================================================
# Write demonstration audit event
# =========================================================

demo_audit_record = write_rag_audit_event(
    question=demo_query,
    answer=refined_demo_answer,
    search_results=refined_demo_results
)


# =========================================================
# Save lineage summary
# =========================================================

lineage_summary = {
    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "openlineage_event_file": str(
        LINEAGE_FILE
    ),
    "jobs_recorded": len(
        lineage_runs
    ),
    "events_recorded": len(
        lineage_events
    ),
    "start_events": start_event_count,
    "complete_events": complete_event_count,
    "runs": lineage_runs,
    "audit_log_file": str(
        AUDIT_FILE
    ),
    "demo_audit_id": (
        demo_audit_record["audit_id"]
    )
}

lineage_summary_path = (
    OUTPUT_DIR / "lineage_audit_summary.json"
)

with lineage_summary_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        lineage_summary,
        file,
        ensure_ascii=False,
        indent=2
    )


# =========================================================
# Results
# =========================================================

print(
    "OpenLineage and audit logging completed successfully."
)

print("-" * 68)
print(
    f"Pipeline jobs recorded:         "
    f"{len(lineage_runs)}"
)
print(
    f"Total OpenLineage events:       "
    f"{len(lineage_events)}"
)
print(
    f"START events:                   "
    f"{start_event_count}"
)
print(
    f"COMPLETE events:                "
    f"{complete_event_count}"
)
print(
    f"Audit events written:           "
    f"1"
)
print("-" * 68)

print("\nRecorded lineage jobs:")

for lineage_run in lineage_runs:
    print(
        f"✓ {lineage_run['job_name']}"
        f" | inputs={lineage_run['input_count']}"
        f" | outputs={lineage_run['output_count']}"
    )

print("\nAudit query:")
print(demo_audit_record["question"])

print("\nAudit primary source:")
print(
    "✓",
    demo_audit_record["primary_source"]
)

print("\nSaved files:")
print("✓", LINEAGE_FILE)
print("✓", AUDIT_FILE)
print("✓", lineage_summary_path)

OpenLineage and audit logging completed successfully.
--------------------------------------------------------------------
Pipeline jobs recorded:         5
Total OpenLineage events:       10
START events:                   5
COMPLETE events:                5
Audit events written:           1
--------------------------------------------------------------------

Recorded lineage jobs:
✓ schema_validation_and_bronze_ingestion | inputs=4 | outputs=4
✓ silver_cleaning_and_quality_scoring | inputs=3 | outputs=4
✓ gold_analytics_and_knowledge_corpus | inputs=3 | outputs=4
✓ qdrant_hybrid_rag_indexing | inputs=1 | outputs=3
✓ great_expectations_quality_gate | inputs=3 | outputs=1

Audit query:
My order tracking has not updated for four days. What should I do?

Audit primary source:
✓ Knowledge Base Article KB-0003

Saved files:
✓ /content/customer_support_capstone/logs/lineage/openlineage_events.jsonl
✓ /content/customer_support_capstone/logs/audit/rag_query_audit.jsonl
✓ /content/customer_su

# Airflow End-to-End Orchestration DAG

In [17]:
# =========================================================
# Airflow End-to-End Orchestration DAG
# =========================================================

DAG_FILE = (
    DAG_DIR / "customer_support_pipeline_dag.py"
)

DAG_CONTRACT_FILE = (
    OUTPUT_DIR / "airflow_dag_contract.json"
)


dag_source = '''
"""
End-to-end orchestration DAG for the
Real-Time Customer Support Intelligence Platform.

Airflow 3 TaskFlow DAG.
"""

from datetime import datetime, timezone
from pathlib import Path
import json
import os

from airflow.sdk import dag, task


PROJECT_ROOT = Path(
    os.environ.get(
        "CUSTOMER_SUPPORT_PROJECT_ROOT",
        "/content/customer_support_capstone"
    )
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"
AUDIT_DIR = PROJECT_ROOT / "logs" / "audit"
LINEAGE_DIR = PROJECT_ROOT / "logs" / "lineage"


def require_file(path: Path) -> Path:
    """Raise an error when a required pipeline artifact is missing."""

    if not path.exists():
        raise FileNotFoundError(
            f"Required pipeline artifact was not found: {path}"
        )

    if path.stat().st_size == 0:
        raise ValueError(
            f"Required pipeline artifact is empty: {path}"
        )

    return path


def load_json(path: Path) -> dict:
    """Load a required JSON artifact."""

    require_file(path)

    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


@dag(
    dag_id="customer_support_intelligence_pipeline",
    description=(
        "End-to-end customer support ingestion, quality, "
        "analytics, RAG, lineage, and audit workflow."
    ),
    schedule=None,
    start_date=datetime(
        2025,
        1,
        1,
        tzinfo=timezone.utc
    ),
    catchup=False,
    max_active_runs=1,
    default_args={
        "owner": "customer-support-team",
        "retries": 1
    },
    tags=[
        "customer-support",
        "quality",
        "rag",
        "qdrant",
        "openlineage"
    ]
)
def customer_support_intelligence_pipeline():

    @task
    def validate_bronze_ingestion() -> dict:
        """
        Confirm schema validation, Bronze ingestion,
        and quarantine routing.
        """

        report = load_json(
            OUTPUT_DIR / "validation_report.json"
        )

        valid_records = report.get(
            "total_valid_records",
            0
        )

        quarantined_records = report.get(
            "total_quarantined_records",
            0
        )

        if valid_records <= 0:
            raise ValueError(
                "Bronze ingestion contains no valid records."
            )

        required_bronze_files = [
            PROJECT_ROOT
            / "data/bronze/support_tickets_bronze.jsonl",

            PROJECT_ROOT
            / "data/bronze/chat_messages_bronze.jsonl",

            PROJECT_ROOT
            / "data/bronze/knowledge_articles_bronze.jsonl"
        ]

        for file_path in required_bronze_files:
            require_file(file_path)

        return {
            "stage": "bronze",
            "status": "passed",
            "valid_records": valid_records,
            "quarantined_records": quarantined_records
        }


    @task
    def validate_silver_transformation(
        bronze_result: dict
    ) -> dict:
        """
        Confirm cleaning, PII masking, deduplication,
        and response quality scoring.
        """

        if bronze_result.get("status") != "passed":
            raise RuntimeError(
                "Bronze stage did not pass."
            )

        report = load_json(
            OUTPUT_DIR / "silver_quality_report.json"
        )

        required_silver_files = [
            PROJECT_ROOT
            / "data/silver/support_tickets_silver.jsonl",

            PROJECT_ROOT
            / "data/silver/chat_messages_silver.jsonl",

            PROJECT_ROOT
            / "data/silver/knowledge_articles_silver.jsonl",

            OUTPUT_DIR / "low_quality_responses.jsonl"
        ]

        for file_path in required_silver_files:
            require_file(file_path)

        quality_distribution = report.get(
            "response_quality",
            {}
        )

        return {
            "stage": "silver",
            "status": "passed",
            "ticket_records": report.get(
                "ticket_records",
                0
            ),
            "low_quality_responses": (
                quality_distribution.get("low", 0)
            ),
            "pii_masking_applied": report.get(
                "pii_masking_applied",
                False
            )
        }


    @task
    def validate_gold_layer(
        silver_result: dict
    ) -> dict:
        """
        Confirm Gold analytics, KPI generation,
        contradiction checks, and RAG corpus creation.
        """

        if silver_result.get("status") != "passed":
            raise RuntimeError(
                "Silver stage did not pass."
            )

        summary = load_json(
            OUTPUT_DIR / "gold_summary_report.json"
        )

        kpis = load_json(
            GOLD_DIR / "support_kpis_gold.json"
        )

        required_gold_files = [
            GOLD_DIR / "support_tickets_gold.jsonl",
            GOLD_DIR / "knowledge_corpus_gold.jsonl",
            GOLD_DIR / "contradiction_alerts.jsonl",
            GOLD_DIR / "support_kpis_gold.json"
        ]

        for file_path in required_gold_files:
            require_file(file_path)

        corpus_documents = summary.get(
            "rag_corpus_documents",
            0
        )

        if corpus_documents <= 0:
            raise ValueError(
                "The Gold RAG corpus contains no documents."
            )

        return {
            "stage": "gold",
            "status": "passed",
            "gold_ticket_records": summary.get(
                "gold_ticket_records",
                0
            ),
            "rag_corpus_documents": corpus_documents,
            "contradiction_alerts": summary.get(
                "contradiction_alerts",
                0
            ),
            "sla_compliance_rate": kpis.get(
                "sla_compliance_rate"
            )
        }


    @task
    def enforce_quality_gate(
        gold_result: dict
    ) -> dict:
        """
        Stop the pipeline when Great Expectations fails.
        """

        if gold_result.get("status") != "passed":
            raise RuntimeError(
                "Gold stage did not pass."
            )

        quality_report = load_json(
            OUTPUT_DIR
            / "great_expectations"
            / "great_expectations_quality_gate.json"
        )

        quality_gate_passed = quality_report.get(
            "quality_gate_passed",
            False
        )

        if not quality_gate_passed:
            raise RuntimeError(
                "Great Expectations quality gate failed. "
                "RAG indexing has been blocked."
            )

        suites = quality_report.get(
            "suites",
            {}
        )

        failed_suites = [
            suite_name
            for suite_name, suite_result in suites.items()
            if not suite_result.get("success", False)
        ]

        if failed_suites:
            raise RuntimeError(
                "Failed expectation suites: "
                + ", ".join(failed_suites)
            )

        return {
            "stage": "quality_gate",
            "status": "passed",
            "great_expectations_version": (
                quality_report.get(
                    "great_expectations_version"
                )
            ),
            "validated_suites": len(suites)
        }


    @task
    def validate_rag_index(
        quality_result: dict
    ) -> dict:
        """
        Confirm Qdrant indexing, hybrid retrieval,
        reranking, and citation output.
        """

        if quality_result.get("status") != "passed":
            raise RuntimeError(
                "Quality gate did not pass."
            )

        manifest = load_json(
            OUTPUT_DIR / "rag_index_manifest.json"
        )

        demonstration = load_json(
            OUTPUT_DIR / "rag_demo_result.json"
        )

        indexed_chunks = manifest.get(
            "indexed_chunks",
            0
        )

        if indexed_chunks <= 0:
            raise ValueError(
                "The RAG index contains no chunks."
            )

        citations = demonstration.get(
            "citations",
            []
        )

        if not citations:
            raise ValueError(
                "The RAG demonstration contains no citations."
            )

        primary_source = demonstration.get(
            "primary_source"
        )

        if not primary_source:
            raise ValueError(
                "The RAG response has no primary source."
            )

        return {
            "stage": "rag",
            "status": "passed",
            "collection_name": manifest.get(
                "collection_name"
            ),
            "indexed_chunks": indexed_chunks,
            "embedding_model": manifest.get(
                "embedding_model"
            ),
            "reranker_model": manifest.get(
                "reranker_model"
            ),
            "primary_source": primary_source,
            "citation_count": len(citations)
        }


    @task
    def validate_lineage_and_audit(
        rag_result: dict
    ) -> dict:
        """
        Confirm OpenLineage START/COMPLETE events
        and query audit records.
        """

        if rag_result.get("status") != "passed":
            raise RuntimeError(
                "RAG stage did not pass."
            )

        lineage_summary = load_json(
            OUTPUT_DIR / "lineage_audit_summary.json"
        )

        lineage_file = require_file(
            LINEAGE_DIR / "openlineage_events.jsonl"
        )

        audit_file = require_file(
            AUDIT_DIR / "rag_query_audit.jsonl"
        )

        start_events = lineage_summary.get(
            "start_events",
            0
        )

        complete_events = lineage_summary.get(
            "complete_events",
            0
        )

        if start_events == 0:
            raise ValueError(
                "No OpenLineage START events were recorded."
            )

        if complete_events == 0:
            raise ValueError(
                "No OpenLineage COMPLETE events were recorded."
            )

        if start_events != complete_events:
            raise ValueError(
                "OpenLineage START and COMPLETE event "
                "counts do not match."
            )

        return {
            "stage": "lineage_and_audit",
            "status": "passed",
            "jobs_recorded": lineage_summary.get(
                "jobs_recorded",
                0
            ),
            "start_events": start_events,
            "complete_events": complete_events,
            "lineage_file": str(lineage_file),
            "audit_file": str(audit_file)
        }


    @task
    def publish_pipeline_summary(
        bronze_result: dict,
        silver_result: dict,
        gold_result: dict,
        quality_result: dict,
        rag_result: dict,
        lineage_result: dict
    ) -> dict:
        """
        Publish the final orchestration result.
        """

        stage_results = [
            bronze_result,
            silver_result,
            gold_result,
            quality_result,
            rag_result,
            lineage_result
        ]

        pipeline_passed = all(
            result.get("status") == "passed"
            for result in stage_results
        )

        final_summary = {
            "dag_id": (
                "customer_support_intelligence_pipeline"
            ),
            "run_completed_at": datetime.now(
                timezone.utc
            ).isoformat(),
            "pipeline_status": (
                "passed"
                if pipeline_passed
                else "failed"
            ),
            "stages": stage_results
        }

        summary_path = (
            OUTPUT_DIR / "airflow_run_summary.json"
        )

        summary_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with summary_path.open(
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(
                final_summary,
                file,
                ensure_ascii=False,
                indent=2
            )

        if not pipeline_passed:
            raise RuntimeError(
                "One or more pipeline stages failed."
            )

        return final_summary


    bronze_result = validate_bronze_ingestion()

    silver_result = validate_silver_transformation(
        bronze_result
    )

    gold_result = validate_gold_layer(
        silver_result
    )

    quality_result = enforce_quality_gate(
        gold_result
    )

    rag_result = validate_rag_index(
        quality_result
    )

    lineage_result = validate_lineage_and_audit(
        rag_result
    )

    publish_pipeline_summary(
        bronze_result=bronze_result,
        silver_result=silver_result,
        gold_result=gold_result,
        quality_result=quality_result,
        rag_result=rag_result,
        lineage_result=lineage_result
    )


customer_support_intelligence_pipeline()
'''.strip() + "\n"


# =========================================================
# Validate Python syntax before saving
# =========================================================

compile(
    dag_source,
    str(DAG_FILE),
    "exec"
)


# =========================================================
# Save DAG file
# =========================================================

DAG_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

with DAG_FILE.open(
    "w",
    encoding="utf-8"
) as file:
    file.write(dag_source)


# =========================================================
# Save orchestration contract
# =========================================================

dag_contract = {
    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),

    "dag_id": (
        "customer_support_intelligence_pipeline"
    ),

    "airflow_api": "Airflow 3 TaskFlow API",

    "schedule": None,

    "catchup": False,

    "tasks": [
        "validate_bronze_ingestion",
        "validate_silver_transformation",
        "validate_gold_layer",
        "enforce_quality_gate",
        "validate_rag_index",
        "validate_lineage_and_audit",
        "publish_pipeline_summary"
    ],

    "dependencies": [
        [
            "validate_bronze_ingestion",
            "validate_silver_transformation"
        ],
        [
            "validate_silver_transformation",
            "validate_gold_layer"
        ],
        [
            "validate_gold_layer",
            "enforce_quality_gate"
        ],
        [
            "enforce_quality_gate",
            "validate_rag_index"
        ],
        [
            "validate_rag_index",
            "validate_lineage_and_audit"
        ],
        [
            "validate_lineage_and_audit",
            "publish_pipeline_summary"
        ]
    ],

    "quality_gate_behavior": (
        "Blocks RAG indexing when Great Expectations fails"
    ),

    "dag_file": str(DAG_FILE),

    "syntax_validation": "passed"
}


with DAG_CONTRACT_FILE.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        dag_contract,
        file,
        ensure_ascii=False,
        indent=2
    )


# =========================================================
# Validate required stage artifacts
# =========================================================

dag_required_artifacts = [
    OUTPUT_DIR / "validation_report.json",
    OUTPUT_DIR / "silver_quality_report.json",
    OUTPUT_DIR / "gold_summary_report.json",
    OUTPUT_DIR
    / "great_expectations"
    / "great_expectations_quality_gate.json",
    OUTPUT_DIR / "rag_index_manifest.json",
    OUTPUT_DIR / "rag_demo_result.json",
    OUTPUT_DIR / "lineage_audit_summary.json"
]

available_artifacts = []
missing_artifacts = []

for artifact_path in dag_required_artifacts:
    if (
        artifact_path.exists()
        and artifact_path.stat().st_size > 0
    ):
        available_artifacts.append(
            str(artifact_path)
        )
    else:
        missing_artifacts.append(
            str(artifact_path)
        )


# =========================================================
# Results
# =========================================================

print("Airflow DAG created successfully.")
print("-" * 68)
print(
    f"DAG ID:                         "
    f"{dag_contract['dag_id']}"
)
print(
    f"Task count:                     "
    f"{len(dag_contract['tasks'])}"
)
print(
    f"Dependency count:               "
    f"{len(dag_contract['dependencies'])}"
)
print(
    f"Python syntax validation:       "
    f"{dag_contract['syntax_validation'].upper()}"
)
print(
    f"Required artifacts available:   "
    f"{len(available_artifacts)}"
)
print(
    f"Required artifacts missing:     "
    f"{len(missing_artifacts)}"
)
print("-" * 68)

print("\nTask sequence:")

for task_number, task_name in enumerate(
    dag_contract["tasks"],
    start=1
):
    print(
        f"{task_number}. {task_name}"
    )

print("\nQuality gate behavior:")
print(
    "✓ Great Expectations failure blocks RAG indexing"
)

print("\nSaved files:")
print("✓", DAG_FILE)
print("✓", DAG_CONTRACT_FILE)

if missing_artifacts:
    print("\nMissing artifacts:")

    for artifact_path in missing_artifacts:
        print("✗", artifact_path)

Airflow DAG created successfully.
--------------------------------------------------------------------
DAG ID:                         customer_support_intelligence_pipeline
Task count:                     7
Dependency count:               6
Python syntax validation:       PASSED
Required artifacts available:   7
Required artifacts missing:     0
--------------------------------------------------------------------

Task sequence:
1. validate_bronze_ingestion
2. validate_silver_transformation
3. validate_gold_layer
4. enforce_quality_gate
5. validate_rag_index
6. validate_lineage_and_audit
7. publish_pipeline_summary

Quality gate behavior:
✓ Great Expectations failure blocks RAG indexing

Saved files:
✓ /content/customer_support_capstone/dags/customer_support_pipeline_dag.py
✓ /content/customer_support_capstone/outputs/airflow_dag_contract.json


# Final Documentation & Submission Package

In [18]:
# =========================================================
# Final Documentation & Submission Package
# =========================================================

README_PATH = PROJECT_ROOT / "README.md"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"
MANIFEST_PATH = PROJECT_ROOT / "project_manifest.json"
PROJECT_TREE_PATH = PROJECT_ROOT / "project_tree.txt"

SUBMISSION_ZIP = Path(
    "/content/customer_support_capstone_submission.zip"
)


# =========================================================
# README
# =========================================================

readme_content = """
# Real-Time Customer Support Intelligence Platform

## Project Overview

This project is an academic prototype for processing customer-support
tickets, chat messages, and knowledge-base articles.

The pipeline validates incoming records, separates invalid records,
creates Bronze, Silver, and Gold datasets, evaluates response quality,
builds a hybrid RAG index, generates cited support answers, validates
data quality, and records lineage and audit events.

## Implemented Pipeline

```text
Raw Support Data
        |
        v
JSON Schema Validation
        |
        +------ Invalid Records ------> Quarantine
        |
        v
Bronze Layer
        |
        v
Silver Cleaning, PII Masking and Quality Scoring
        |
        v
Gold Analytics, KPIs and Knowledge Corpus
        |
        v
Great Expectations Quality Gate
        |
        v
Qdrant Dense Search + BM25
        |
        v
Reciprocal Rank Fusion
        |
        v
Cross-Encoder Reranking
        |
        v
Grounded Answer with Citations
        |
        +------> OpenLineage Events
        |
        +------> RAG Query Audit Log

SyntaxError: incomplete input (3452760136.py, line 19)

In [19]:
from google.colab import userdata

github_token = userdata.get("GITHUB_TOKEN")

print("GitHub token connected:", bool(github_token))

GitHub token connected: True


In [20]:
# =========================================================
# GitHub Repository Configuration
# =========================================================

GITHUB_USERNAME = "khalidalzzahranii-eng"
REPOSITORY_NAME = "customer-support-intelligence-platform-sdaia-"

GITHUB_REPOSITORY_URL = (
    f"https://github.com/"
    f"{GITHUB_USERNAME}/{REPOSITORY_NAME}.git"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder was not found: {PROJECT_ROOT}"
    )

project_files = [
    path
    for path in PROJECT_ROOT.rglob("*")
    if path.is_file()
]

print("GitHub configuration is ready.")
print("-" * 60)
print("Repository:", GITHUB_REPOSITORY_URL)
print("Project folder:", PROJECT_ROOT)
print("Project files found:", len(project_files))

GitHub configuration is ready.
------------------------------------------------------------
Repository: https://github.com/khalidalzzahranii-eng/customer-support-intelligence-platform-sdaia-.git
Project folder: /content/customer_support_capstone
Project files found: 36


In [21]:
# =========================================================
# Create .gitignore
# =========================================================

gitignore_content = """
# Local Qdrant database is rebuilt by the notebook
artifacts/vector_db/

# Python temporary files
__pycache__/
*.pyc
*.pyo

# Jupyter temporary files
.ipynb_checkpoints/

# Archives
*.zip

# Secrets and environment files
.env
*.key

# Operating-system files
.DS_Store
Thumbs.db
""".strip() + "\n"

GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"

GITIGNORE_PATH.write_text(
    gitignore_content,
    encoding="utf-8"
)

print(".gitignore created successfully.")
print("✓", GITIGNORE_PATH)
print("\nContents:")
print(GITIGNORE_PATH.read_text(encoding="utf-8"))

.gitignore created successfully.
✓ /content/customer_support_capstone/.gitignore

Contents:
# Local Qdrant database is rebuilt by the notebook
artifacts/vector_db/

# Python temporary files
__pycache__/
*.pyc
*.pyo

# Jupyter temporary files
.ipynb_checkpoints/

# Archives
*.zip

# Secrets and environment files
.env
*.key

# Operating-system files
.DS_Store
Thumbs.db



In [22]:
# =========================================================
# Initialize Git Repository
# =========================================================

def run_git_command(arguments, check=True):
    result = subprocess.run(
        ["git", *arguments],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            result.stderr.strip()
            or f"Git command failed: {' '.join(arguments)}"
        )

    return result


# تهيئة Git داخل مجلد المشروع
run_git_command(["init"])

# بيانات صاحب المستودع
run_git_command([
    "config",
    "user.name",
    GITHUB_USERNAME
])

run_git_command([
    "config",
    "user.email",
    f"{GITHUB_USERNAME}@users.noreply.github.com"
])

# اعتماد الفرع الرئيسي main
run_git_command([
    "branch",
    "-M",
    "main"
])

# إضافة رابط المستودع أو تحديثه إذا كان موجودًا
remote_result = run_git_command(
    ["remote", "get-url", "origin"],
    check=False
)

if remote_result.returncode == 0:
    run_git_command([
        "remote",
        "set-url",
        "origin",
        GITHUB_REPOSITORY_URL
    ])
else:
    run_git_command([
        "remote",
        "add",
        "origin",
        GITHUB_REPOSITORY_URL
    ])

configured_remote = run_git_command([
    "remote",
    "get-url",
    "origin"
]).stdout.strip()

print("Git repository initialized successfully.")
print("-" * 60)
print("Branch: main")
print("Remote:", configured_remote)
print("Project:", PROJECT_ROOT)

Git repository initialized successfully.
------------------------------------------------------------
Branch: main
Remote: https://github.com/khalidalzzahranii-eng/customer-support-intelligence-platform-sdaia-.git
Project: /content/customer_support_capstone


In [23]:
# =========================================================
# Stage and Commit Project Files
# =========================================================

run_git_command(["add", "."])

status_result = run_git_command(
    ["status", "--short"],
    check=False
)

print("Files prepared for commit:")
print("-" * 60)

if status_result.stdout.strip():
    print(status_result.stdout.strip())

    commit_result = run_git_command([
        "commit",
        "-m",
        "Initial capstone project submission"
    ])

    print("\nCommit created successfully.")
    print(commit_result.stdout.strip())
else:
    print("No new files or changes were found.")

Files prepared for commit:
------------------------------------------------------------
A  .gitignore
A  dags/customer_support_pipeline_dag.py
A  data/bronze/chat_messages_bronze.jsonl
A  data/bronze/knowledge_articles_bronze.jsonl
A  data/bronze/support_tickets_bronze.jsonl
A  data/gold/contradiction_alerts.jsonl
A  data/gold/knowledge_corpus_gold.jsonl
A  data/gold/support_kpis_gold.json
A  data/gold/support_tickets_gold.jsonl
A  data/quarantine/schema_validation_failures.jsonl
A  data/raw/chat_messages.jsonl
A  data/raw/invalid_records.jsonl
A  data/raw/knowledge_articles.jsonl
A  data/raw/support_tickets.jsonl
A  data/silver/chat_messages_silver.jsonl
A  data/silver/knowledge_articles_silver.jsonl
A  data/silver/support_tickets_silver.jsonl
A  logs/audit/rag_query_audit.jsonl
A  logs/lineage/openlineage_events.jsonl
A  outputs/airflow_dag_contract.json
A  outputs/gold_summary_report.json
A  outputs/great_expectations/chat_messages_suite_20260706_225457_result.json
A  outputs/great_

In [24]:
# =========================================================
# Create Repository Documentation
# =========================================================

README_PATH = PROJECT_ROOT / "README.md"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"
MANIFEST_PATH = PROJECT_ROOT / "project_manifest.json"


readme_lines = [
    "# Real-Time Customer Support Intelligence Platform",
    "",
    "## Overview",
    "",
    "A Google Colab academic prototype for customer-support data processing,",
    "quality monitoring, hybrid retrieval-augmented generation, lineage,",
    "and workflow orchestration.",
    "",
    "## Pipeline",
    "",
    "```text",
    "Raw JSONL Data",
    "    -> JSON Schema Validation",
    "    -> Bronze Layer / Quarantine",
    "    -> Silver Cleaning, PII Masking and Quality Scoring",
    "    -> Gold Analytics and Knowledge Corpus",
    "    -> Great Expectations Quality Gate",
    "    -> Qdrant Dense Search + BM25",
    "    -> Reciprocal Rank Fusion",
    "    -> Cross-Encoder Reranking",
    "    -> Grounded Answer with Citations",
    "```",
    "",
    "## Implemented Features",
    "",
    "- JSON Schema validation",
    "- Invalid-record quarantine",
    "- Bronze, Silver and Gold logical layers",
    "- Text normalization and PII masking",
    "- Deduplication",
    "- Agent-response quality scoring",
    "- SLA and KPI calculation",
    "- Context-aware contradiction detection",
    "- Sentence Transformer embeddings",
    "- Qdrant local vector index",
    "- BM25 lexical search",
    "- Reciprocal Rank Fusion",
    "- Cross-encoder reranking",
    "- Grounded answers with citations",
    "- Great Expectations validation suites",
    "- OpenLineage START and COMPLETE events",
    "- RAG query audit logs",
    "- Airflow TaskFlow DAG",
    "",
    "## Demonstration Results",
    "",
    "- Valid Bronze records: 70",
    "- Quarantined records: 3",
    "- High-quality responses: 27",
    "- Low-quality responses: 3",
    "- RAG corpus documents: 37",
    "- Indexed chunks: 37",
    "- Great Expectations: PASSED",
    "- OpenLineage events: 10",
    "- Airflow DAG tasks: 7",
    "",
    "## Example Query",
    "",
    "**Question:**",
    "",
    "My order tracking has not updated for four days. What should I do?",
    "",
    "**Primary source:** `Knowledge Base Article KB-0003`",
    "",
    "## Running the Project",
    "",
    "1. Open `Customer_Support_Capstone.ipynb` in Google Colab.",
    "2. Select a hosted Python runtime.",
    "3. Run all cells from top to bottom.",
    "4. Wait for the embedding and reranker models to download.",
    "5. Review the generated files under `/content/customer_support_capstone`.",
    "",
    "## Important Files",
    "",
    "- `Customer_Support_Capstone.ipynb`",
    "- `dags/customer_support_pipeline_dag.py`",
    "- `outputs/validation_report.json`",
    "- `outputs/silver_quality_report.json`",
    "- `outputs/gold_summary_report.json`",
    "- `outputs/rag_demo_result.json`",
    "- `outputs/great_expectations/great_expectations_quality_gate.json`",
    "- `logs/lineage/openlineage_events.jsonl`",
    "- `logs/audit/rag_query_audit.jsonl`",
    "",
    "## Prototype Scope",
    "",
    "The Colab version uses JSONL ingestion and Qdrant local mode.",
    "The Airflow DAG is generated and syntax-validated but requires an",
    "Airflow environment for scheduler execution.",
    "",
    "Kafka, Flink and Delta Lake are production architecture extensions",
    "and are not active services in this Colab prototype.",
]

README_PATH.write_text(
    "\n".join(readme_lines) + "\n",
    encoding="utf-8"
)


requirements_lines = [
    "jsonschema",
    "pandas",
    "numpy",
    "qdrant-client",
    "sentence-transformers",
    "great_expectations==1.18.2",
    "openlineage-python==1.50.0",
    "",
    "# Required only for executing the generated DAG:",
    "# apache-airflow>=3.0.0",
]

REQUIREMENTS_PATH.write_text(
    "\n".join(requirements_lines) + "\n",
    encoding="utf-8"
)


project_manifest = {
    "project_name": (
        "Real-Time Customer Support Intelligence Platform"
    ),
    "execution_environment": "Google Colab",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "implemented_components": [
        "JSON Schema validation",
        "Quarantine routing",
        "Bronze, Silver and Gold layers",
        "PII masking",
        "Response quality scoring",
        "SLA analytics",
        "Contradiction detection",
        "Qdrant vector retrieval",
        "BM25",
        "Reciprocal Rank Fusion",
        "Cross-encoder reranking",
        "Grounded citations",
        "Great Expectations",
        "OpenLineage",
        "Audit logging",
        "Airflow DAG"
    ],
    "production_extensions_not_active": [
        "Kafka",
        "Flink",
        "Delta Lake",
        "Airflow scheduler"
    ]
}

with MANIFEST_PATH.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        project_manifest,
        file,
        ensure_ascii=False,
        indent=2
    )


print("Repository documentation created successfully.")
print("✓", README_PATH)
print("✓", REQUIREMENTS_PATH)
print("✓", MANIFEST_PATH)

Repository documentation created successfully.
✓ /content/customer_support_capstone/README.md
✓ /content/customer_support_capstone/requirements.txt
✓ /content/customer_support_capstone/project_manifest.json


In [25]:
# =========================================================
# Commit Repository Documentation
# =========================================================

run_git_command([
    "add",
    "README.md",
    "requirements.txt",
    "project_manifest.json"
])

documentation_status = run_git_command(
    ["status", "--short"],
    check=False
)

print("Documentation changes:")
print("-" * 60)
print(documentation_status.stdout.strip())

if documentation_status.stdout.strip():
    commit_result = run_git_command([
        "commit",
        "-m",
        "Add project documentation and requirements"
    ])

    print("\nDocumentation commit created successfully.")
    print(commit_result.stdout.strip())
else:
    print("No documentation changes were found.")

Documentation changes:
------------------------------------------------------------
A  README.md
A  project_manifest.json
A  requirements.txt

Documentation commit created successfully.
[main f6e50ee] Add project documentation and requirements
 3 files changed, 131 insertions(+)
 create mode 100644 README.md
 create mode 100644 project_manifest.json
 create mode 100644 requirements.txt


In [27]:
# =========================================================
# Push Commits to GitHub
# =========================================================

import base64
from google.colab import userdata

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )

authentication_value = base64.b64encode(
    f"{GITHUB_USERNAME}:{github_token}".encode("utf-8")
).decode("utf-8")

push_result = subprocess.run(
    [
        "git",
        "-c",
        (
            "http.extraHeader="
            f"Authorization: Basic {authentication_value}"
        ),
        "push",
        "-u",
        "origin",
        "main"
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True
)

# حذف القيم الحساسة من الذاكرة
del authentication_value
del github_token

if push_result.returncode != 0:
    raise RuntimeError(
        "GitHub push failed:\n"
        + push_result.stderr.strip()
    )

print("Project files uploaded to GitHub successfully.")
print("-" * 60)
print(push_result.stdout.strip())
print(push_result.stderr.strip())
print("\nRepository:")
print(GITHUB_REPOSITORY_URL)

RuntimeError: GitHub push failed:
remote: Permission to khalidalzzahranii-eng/customer-support-intelligence-platform-sdaia-.git denied to khalidalzzahranii-eng.
fatal: unable to access 'https://github.com/khalidalzzahranii-eng/customer-support-intelligence-platform-sdaia-.git/': The requested URL returned error: 403

In [28]:
# =========================================================
# Retry GitHub Push
# =========================================================

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )

authentication_value = base64.b64encode(
    f"{GITHUB_USERNAME}:{github_token}".encode("utf-8")
).decode("utf-8")

push_result = subprocess.run(
    [
        "git",
        "-c",
        (
            "http.extraHeader="
            f"Authorization: Basic {authentication_value}"
        ),
        "push",
        "-u",
        "origin",
        "main"
    ],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True
)

del authentication_value
del github_token

if push_result.returncode != 0:
    raise RuntimeError(
        "GitHub push failed:\n"
        + push_result.stderr.strip()
    )

print("Project uploaded to GitHub successfully.")
print("-" * 60)
print(push_result.stderr.strip())
print("\nRepository:")
print(GITHUB_REPOSITORY_URL)

Project uploaded to GitHub successfully.
------------------------------------------------------------
To https://github.com/khalidalzzahranii-eng/customer-support-intelligence-platform-sdaia-.git
 * [new branch]      main -> main

Repository:
https://github.com/khalidalzzahranii-eng/customer-support-intelligence-platform-sdaia-.git
